# 🕵️ Dia 49 — Manuseio de evidência: hash e cadeia de custódia

**Investigador Cyber 90 Dias** · Fase 4 · Rank Jutsu `B`

> Baseado no *Day 49* de [cyber-investigator-skills](https://github.com/thiefdroppers/cyber-investigator-skills/blob/main/curriculum/phase-4-digital-forensics-incident-investigation/day-49.md), © ThiefDroppers (MIT).
> Notebook, camada BR e Camada 8 por Deivison Lourenço ([@Ridd1kulusC0d3r](https://github.com/Ridd1kulusC0d3r)).

---

### 🎬 Onde paramos

5 de outubro, 16h05 em Brasília. A TI da Serra Azul entrega um pen drive lacrado (`SA-0001`) com os logs da manhã em que a caixa da agência Centro clicou no link no notebook `SA-CX-07`. Rafa segura seu pulso: *"Primeiro o formulário. Depois o hash. Só depois a análise."*

Neste notebook você vai:

1. **Montar `evidence/`, `work/` e `notes/`** com o EVID-001 do caso LAB-P4 (o laboratório do original).
2. **Gerar e conferir o manifesto SHA-256**, como `sha256sum` e `sha256sum -c` fazem.
3. **Ver um único espaço quebrar o hash**, e documentar o erro.
4. **Repetir tudo com o EVID-SA-001** da Serra Azul.
5. **Jogar o quiz das 10 etapas do CPP** (art. 158-B) e da ordem de volatilidade.
6. **Preencher o formulário de custódia** (Partes A a E) e sair com o `.md` e o hash dele.

🐍 Só biblioteca padrão. Rode de cima para baixo. Sem internet? Funciona igual: os arquivos vêm embutidos.


## 1. Receber o EVID-001

O EVID-001 é o pacote de logs sintético do LAB-P4: `bastion01-auth.log`, `fs01-auth.log`, `docportal-app.jsonl` e `firewall.log`. A célula tenta baixar do GitHub do original; se não conseguir, usa a cópia embutida (o arquivo inteiro, não um resumo, então o hash é o mesmo).


In [ ]:
# ------------------------------------------------------------
# RECEBER O EVID-001 (PACOTE DE LOGS DO LAB-P4)
# Plano A: baixar do repositório original (GitHub raw), com urllib.
# Plano B: usar a cópia embutida logo abaixo (o arquivo inteiro,
#          copiado do upstream/), para o notebook rodar sem internet.
# ------------------------------------------------------------
import urllib.request      # baixa arquivos da web, sem instalar nada
import hashlib             # SHA-256: a impressão digital dos bytes

FORCAR_OFFLINE = False     # True = nem tenta a rede (útil em sala sem Wi-Fi)
URL_BASE = 'https://raw.githubusercontent.com/thiefdroppers/cyber-investigator-skills/main/curriculum/phase-4-digital-forensics-incident-investigation/resources/'

RESERVA = {}             # caminho do arquivo -> conteúdo embutido (plano B)
RESERVA['case-lab-p4/logs/bastion01-auth.log'] = ''.join((
    'Mar 13 14:02:11 bastion01 sshd[21807]: Accepted publickey for ops.admin from 10.10.50.12 port 51544 ssh2: ED25519 SHA256:q8m1Xx0lab0synthetic0key0fingerprint0AAAA\n'
    'Mar 13 14:02:12 bastion01 sshd[21807]: pam_unix(sshd:session): session opened for user ops.admin(uid=1001) by (uid=0)\n'
    'Mar 13 16:47:39 bastion01 sshd[21814]: Accepted publickey for ops.admin from 10.10.50.12 port 52010 ssh2: ED25519 SHA256:q8m1Xx0lab0synthetic0key0fingerprint0AAAA\n'
    'Mar 13 16:47:40 bastion01 sshd[21814]: pam_unix(sshd:session): session opened for user ops.admin(uid=1001) by (uid=0)\n'
    'Mar 14 02:51:03 bastion01 sshd[21815]: Failed password for invalid user admin from 203.0.113.45 port 49208 ssh2\n'
    'Mar 14 02:51:15 bastion01 sshd[21816]: Failed password for root from 203.0.113.45 port 49212 ssh2\n'
    'Mar 14 02:51:24 bastion01 sshd[21817]: Failed password for invalid user backup from 203.0.113.45 port 49216 ssh2\n'
    'Mar 14 02:51:34 bastion01 sshd[21818]: Failed password for invalid user oracle from 203.0.113.45 port 49224 ssh2\n'
    'Mar 14 02:51:49 bastion01 sshd[21819]: Failed password for svc_backup from 203.0.113.45 port 49230 ssh2\n'
    'Mar 14 02:52:04 bastion01 sshd[21820]: Failed password for invalid user test from 203.0.113.45 port 49237 ssh2\n'
    'Mar 14 02:52:18 bastion01 sshd[21821]: Failed password for invalid user ubuntu from 203.0.113.45 port 49245 ssh2\n'
    'Mar 14 02:52:30 bastion01 sshd[21822]: Failed password for invalid user ftpuser from 203.0.113.45 port 49249 ssh2\n'
    'Mar 14 02:52:39 bastion01 sshd[21823]: Failed password for invalid user admin from 203.0.113.45 port 49252 ssh2\n'
    'Mar 14 02:52:52 bastion01 sshd[21824]: Failed password for root from 203.0.113.45 port 49256 ssh2\n'
    'Mar 14 02:53:02 bastion01 sshd[21825]: Failed password for invalid user backup from 203.0.113.45 port 49264 ssh2\n'
    'Mar 14 02:53:16 bastion01 sshd[21826]: Failed password for invalid user oracle from 203.0.113.45 port 49268 ssh2\n'
    'Mar 14 02:53:29 bastion01 sshd[21827]: Failed password for svc_backup from 203.0.113.45 port 49269 ssh2\n'
    'Mar 14 02:53:42 bastion01 sshd[21828]: Failed password for invalid user test from 203.0.113.45 port 49278 ssh2\n'
    'Mar 14 02:53:51 bastion01 sshd[21829]: Failed password for invalid user ubuntu from 203.0.113.45 port 49279 ssh2\n'
    'Mar 14 02:54:03 bastion01 sshd[21830]: Failed password for invalid user ftpuser from 203.0.113.45 port 49287 ssh2\n'
    'Mar 14 02:54:18 bastion01 sshd[21831]: Failed password for invalid user admin from 203.0.113.45 port 49296 ssh2\n'
    'Mar 14 02:54:32 bastion01 sshd[21832]: Failed password for root from 203.0.113.45 port 49303 ssh2\n'
    'Mar 14 02:54:43 bastion01 sshd[21833]: Failed password for invalid user backup from 203.0.113.45 port 49311 ssh2\n'
    'Mar 14 02:54:53 bastion01 sshd[21834]: Failed password for invalid user oracle from 203.0.113.45 port 49316 ssh2\n'
    'Mar 14 02:55:04 bastion01 sshd[21835]: Failed password for svc_backup from 203.0.113.45 port 49321 ssh2\n'
    'Mar 14 02:55:16 bastion01 sshd[21836]: Failed password for invalid user test from 203.0.113.45 port 49330 ssh2\n'
    'Mar 14 02:55:25 bastion01 sshd[21837]: Failed password for invalid user ubuntu from 203.0.113.45 port 49337 ssh2\n'
    'Mar 14 02:55:37 bastion01 sshd[21838]: Failed password for invalid user ftpuser from 203.0.113.45 port 49343 ssh2\n'
    'Mar 14 02:55:50 bastion01 sshd[21839]: Failed password for invalid user admin from 203.0.113.45 port 49347 ssh2\n'
    'Mar 14 02:56:02 bastion01 sshd[21840]: Failed password for root from 203.0.113.45 port 49348 ssh2\n'
    'Mar 14 02:56:14 bastion01 sshd[21841]: Failed password for invalid user backup from 203.0.113.45 port 49357 ssh2\n'
    'Mar 14 02:56:27 bastion01 sshd[21842]: Failed password for invalid user oracle from 203.0.113.45 port 49359 ssh2\n'
    'Mar 14 02:56:42 bastion01 sshd[21843]: Failed password for svc_backup from 203.0.113.45 port 49368 ssh2\n'
    'Mar 14 02:56:54 bastion01 sshd[21844]: Failed password for invalid user test from 203.0.113.45 port 49374 ssh2\n'
    'Mar 14 02:57:08 bastion01 sshd[21845]: Failed password for invalid user ubuntu from 203.0.113.45 port 49376 ssh2\n'
    'Mar 14 02:57:22 bastion01 sshd[21846]: Failed password for invalid user ftpuser from 203.0.113.45 port 49379 ssh2\n'
    'Mar 14 02:57:34 bastion01 sshd[21847]: Failed password for invalid user admin from 203.0.113.45 port 49383 ssh2\n'
    'Mar 14 02:57:45 bastion01 sshd[21848]: Failed password for root from 203.0.113.45 port 49387 ssh2\n'
    'Mar 14 02:57:59 bastion01 sshd[21849]: Failed password for invalid user backup from 203.0.113.45 port 49395 ssh2\n'
    'Mar 14 02:58:11 bastion01 sshd[21850]: Failed password for invalid user oracle from 203.0.113.45 port 49398 ssh2\n'
    'Mar 14 02:58:26 bastion01 sshd[21851]: Failed password for svc_backup from 203.0.113.45 port 49403 ssh2\n'
    'Mar 14 02:58:38 bastion01 sshd[21852]: Failed password for invalid user test from 203.0.113.45 port 49410 ssh2\n'
    'Mar 14 02:58:49 bastion01 sshd[21853]: Failed password for invalid user ubuntu from 203.0.113.45 port 49411 ssh2\n'
    'Mar 14 02:59:03 bastion01 sshd[21854]: Failed password for invalid user ftpuser from 203.0.113.45 port 49419 ssh2\n'
    'Mar 14 02:59:14 bastion01 sshd[21855]: Failed password for invalid user admin from 203.0.113.45 port 49422 ssh2\n'
    'Mar 14 02:59:28 bastion01 sshd[21856]: Failed password for root from 203.0.113.45 port 49430 ssh2\n'
    'Mar 14 02:59:43 bastion01 sshd[21857]: Failed password for invalid user backup from 203.0.113.45 port 49434 ssh2\n'
    'Mar 14 02:59:54 bastion01 sshd[21858]: Failed password for invalid user oracle from 203.0.113.45 port 49436 ssh2\n'
    'Mar 14 03:00:09 bastion01 sshd[21859]: Failed password for svc_backup from 203.0.113.45 port 49440 ssh2\n'
    'Mar 14 03:00:24 bastion01 sshd[21860]: Failed password for invalid user test from 203.0.113.45 port 49443 ssh2\n'
    'Mar 14 03:00:39 bastion01 sshd[21861]: Failed password for invalid user ubuntu from 203.0.113.45 port 49447 ssh2\n'
    'Mar 14 03:00:48 bastion01 sshd[21862]: Failed password for invalid user ftpuser from 203.0.113.45 port 49450 ssh2\n'
    'Mar 14 03:01:02 bastion01 sshd[21863]: Failed password for invalid user admin from 203.0.113.45 port 49458 ssh2\n'
    'Mar 14 03:01:17 bastion01 sshd[21864]: Failed password for root from 203.0.113.45 port 49461 ssh2\n'
    'Mar 14 03:01:29 bastion01 sshd[21865]: Failed password for invalid user backup from 203.0.113.45 port 49470 ssh2\n'
    'Mar 14 03:01:39 bastion01 sshd[21866]: Failed password for invalid user oracle from 203.0.113.45 port 49478 ssh2\n'
    'Mar 14 03:01:53 bastion01 sshd[21867]: Failed password for svc_backup from 203.0.113.45 port 49486 ssh2\n'
    'Mar 14 03:02:08 bastion01 sshd[21868]: Failed password for invalid user test from 203.0.113.45 port 49491 ssh2\n'
    'Mar 14 03:02:23 bastion01 sshd[21869]: Failed password for invalid user ubuntu from 203.0.113.45 port 49499 ssh2\n'
    'Mar 14 03:02:36 bastion01 sshd[21870]: Failed password for invalid user ftpuser from 203.0.113.45 port 49504 ssh2\n'
    'Mar 14 03:02:51 bastion01 sshd[21871]: Failed password for invalid user admin from 203.0.113.45 port 49511 ssh2\n'
    'Mar 14 03:03:03 bastion01 sshd[21872]: Failed password for root from 203.0.113.45 port 49512 ssh2\n'
    'Mar 14 03:03:13 bastion01 sshd[21873]: Failed password for invalid user backup from 203.0.113.45 port 49517 ssh2\n'
    'Mar 14 03:03:28 bastion01 sshd[21874]: Failed password for invalid user oracle from 203.0.113.45 port 49524 ssh2\n'
    'Mar 14 03:03:40 bastion01 sshd[21875]: Failed password for svc_backup from 203.0.113.45 port 49525 ssh2\n'
    'Mar 14 03:03:55 bastion01 sshd[21876]: Failed password for invalid user test from 203.0.113.45 port 49534 ssh2\n'
    'Mar 14 03:04:05 bastion01 sshd[21877]: Failed password for invalid user ubuntu from 203.0.113.45 port 49539 ssh2\n'
    'Mar 14 03:04:17 bastion01 sshd[21878]: Failed password for invalid user ftpuser from 203.0.113.45 port 49547 ssh2\n'
    'Mar 14 03:04:31 bastion01 sshd[21879]: Failed password for invalid user admin from 203.0.113.45 port 49556 ssh2\n'
    'Mar 14 03:04:46 bastion01 sshd[21880]: Failed password for root from 203.0.113.45 port 49561 ssh2\n'
    'Mar 14 03:05:00 bastion01 sshd[21881]: Failed password for invalid user backup from 203.0.113.45 port 49565 ssh2\n'
    'Mar 14 03:05:14 bastion01 sshd[21882]: Failed password for invalid user oracle from 203.0.113.45 port 49568 ssh2\n'
    'Mar 14 03:05:23 bastion01 sshd[21883]: Failed password for svc_backup from 203.0.113.45 port 49576 ssh2\n'
    'Mar 14 03:05:32 bastion01 sshd[21884]: Failed password for invalid user test from 203.0.113.45 port 49577 ssh2\n'
    'Mar 14 03:05:44 bastion01 sshd[21885]: Failed password for invalid user ubuntu from 203.0.113.45 port 49582 ssh2\n'
    'Mar 14 03:05:55 bastion01 sshd[21886]: Failed password for invalid user ftpuser from 203.0.113.45 port 49589 ssh2\n'
    'Mar 14 03:06:06 bastion01 sshd[21887]: Failed password for invalid user admin from 203.0.113.45 port 49593 ssh2\n'
    'Mar 14 03:06:21 bastion01 sshd[21888]: Failed password for root from 203.0.113.45 port 49598 ssh2\n'
    'Mar 14 03:06:34 bastion01 sshd[21889]: Failed password for invalid user backup from 203.0.113.45 port 49602 ssh2\n'
    'Mar 14 03:06:48 bastion01 sshd[21890]: Failed password for invalid user oracle from 203.0.113.45 port 49605 ssh2\n'
    'Mar 14 03:06:57 bastion01 sshd[21891]: Failed password for svc_backup from 203.0.113.45 port 49614 ssh2\n'
    'Mar 14 03:07:07 bastion01 sshd[21892]: Failed password for invalid user test from 203.0.113.45 port 49617 ssh2\n'
    'Mar 14 03:07:18 bastion01 sshd[21893]: Failed password for invalid user ubuntu from 203.0.113.45 port 49625 ssh2\n'
    'Mar 14 03:07:30 bastion01 sshd[21894]: Failed password for invalid user ftpuser from 203.0.113.45 port 49633 ssh2\n'
    'Mar 14 03:07:39 bastion01 sshd[21895]: Failed password for invalid user admin from 203.0.113.45 port 49636 ssh2\n'
    'Mar 14 03:07:51 bastion01 sshd[21896]: Failed password for root from 203.0.113.45 port 49640 ssh2\n'
    'Mar 14 03:08:01 bastion01 sshd[21897]: Failed password for invalid user backup from 203.0.113.45 port 49647 ssh2\n'
    'Mar 14 03:08:10 bastion01 sshd[21898]: Failed password for invalid user oracle from 203.0.113.45 port 49649 ssh2\n'
    'Mar 14 03:08:19 bastion01 sshd[21899]: Failed password for svc_backup from 203.0.113.45 port 49651 ssh2\n'
    'Mar 14 03:08:28 bastion01 sshd[21900]: Failed password for invalid user test from 203.0.113.45 port 49655 ssh2\n'
    'Mar 14 03:08:42 bastion01 sshd[21901]: Failed password for invalid user ubuntu from 203.0.113.45 port 49657 ssh2\n'
    'Mar 14 03:08:54 bastion01 sshd[21902]: Failed password for invalid user ftpuser from 203.0.113.45 port 49658 ssh2\n'
    'Mar 14 03:09:03 bastion01 sshd[21903]: Failed password for invalid user admin from 203.0.113.45 port 49659 ssh2\n'
    'Mar 14 03:09:14 bastion01 sshd[21904]: Failed password for root from 203.0.113.45 port 49664 ssh2\n'
    'Mar 14 03:09:25 bastion01 sshd[21905]: Failed password for invalid user backup from 203.0.113.45 port 49665 ssh2\n'
    'Mar 14 03:09:37 bastion01 sshd[21906]: Failed password for invalid user oracle from 203.0.113.45 port 49673 ssh2\n'
    'Mar 14 03:09:51 bastion01 sshd[21907]: Failed password for svc_backup from 203.0.113.45 port 49681 ssh2\n'
    'Mar 14 03:10:00 bastion01 sshd[21908]: Failed password for invalid user test from 203.0.113.45 port 49689 ssh2\n'
    'Mar 14 03:10:10 bastion01 sshd[21909]: Failed password for invalid user ubuntu from 203.0.113.45 port 49696 ssh2\n'
    'Mar 14 03:10:19 bastion01 sshd[21910]: Failed password for invalid user ftpuser from 203.0.113.45 port 49703 ssh2\n'
    'Mar 14 03:10:33 bastion01 sshd[21911]: Failed password for invalid user admin from 203.0.113.45 port 49712 ssh2\n'
    'Mar 14 03:10:47 bastion01 sshd[21912]: Failed password for root from 203.0.113.45 port 49713 ssh2\n'
    'Mar 14 03:11:02 bastion01 sshd[21913]: Failed password for invalid user backup from 203.0.113.45 port 49720 ssh2\n'
    'Mar 14 03:11:12 bastion01 sshd[21914]: Failed password for invalid user oracle from 203.0.113.45 port 49727 ssh2\n'
    'Mar 14 03:11:22 bastion01 sshd[21915]: Failed password for svc_backup from 203.0.113.45 port 49734 ssh2\n'
    'Mar 14 03:11:36 bastion01 sshd[21916]: Failed password for invalid user test from 203.0.113.45 port 49742 ssh2\n'
    'Mar 14 03:11:49 bastion01 sshd[21917]: Failed password for invalid user ubuntu from 203.0.113.45 port 49744 ssh2\n'
    'Mar 14 03:12:04 bastion01 sshd[21918]: Failed password for invalid user ftpuser from 203.0.113.45 port 49745 ssh2\n'
    'Mar 14 03:12:13 bastion01 sshd[21919]: Failed password for invalid user admin from 203.0.113.45 port 49751 ssh2\n'
    'Mar 14 03:12:23 bastion01 sshd[21920]: Failed password for root from 203.0.113.45 port 49753 ssh2\n'
    'Mar 14 03:12:37 bastion01 sshd[21921]: Failed password for invalid user backup from 203.0.113.45 port 49755 ssh2\n'
    'Mar 14 03:12:47 bastion01 sshd[21922]: Failed password for invalid user oracle from 203.0.113.45 port 49759 ssh2\n'
    'Mar 14 03:13:01 bastion01 sshd[21923]: Failed password for svc_backup from 203.0.113.45 port 49760 ssh2\n'
    'Mar 14 03:13:15 bastion01 sshd[21924]: Failed password for invalid user test from 203.0.113.45 port 49766 ssh2\n'
    'Mar 14 03:13:24 bastion01 sshd[21925]: Failed password for invalid user ubuntu from 203.0.113.45 port 49774 ssh2\n'
    'Mar 14 03:13:36 bastion01 sshd[21926]: Failed password for invalid user ftpuser from 203.0.113.45 port 49783 ssh2\n'
    'Mar 14 03:14:07 bastion01 sshd[21927]: Accepted password for svc_backup from 203.0.113.45 port 50122 ssh2\n'
    'Mar 14 03:14:08 bastion01 sshd[21927]: pam_unix(sshd:session): session opened for user svc_backup(uid=1107) by (uid=0)\n'
    'Mar 14 03:14:08 bastion01 systemd-logind[642]: New session 318 of user svc_backup.\n'
    'Mar 14 03:52:44 bastion01 sshd[21927]: pam_unix(sshd:session): session closed for user svc_backup\n'
))
RESERVA['case-lab-p4/logs/docportal-app.jsonl'] = ''.join((
    '{"ts": "2026-03-13T13:22:23Z", "level": "INFO", "event": "document.view", "user": "acct.clerk01", "src_ip": "10.10.40.63", "path": "/finance/2026-Q1/inv-1678.pdf", "count": 1}\n'
    '{"ts": "2026-03-13T13:53:23Z", "level": "INFO", "event": "document.view", "user": "acct.lead02", "src_ip": "10.10.40.57", "path": "/finance/2026-Q1/inv-1836.pdf", "count": 1}\n'
    '{"ts": "2026-03-13T14:26:23Z", "level": "INFO", "event": "document.view", "user": "fin.mgr01", "src_ip": "10.10.40.63", "path": "/finance/2026-Q1/inv-1493.pdf", "count": 1}\n'
    '{"ts": "2026-03-13T14:27:23Z", "level": "INFO", "event": "document.view", "user": "fin.mgr01", "src_ip": "10.10.40.61", "path": "/finance/2026-Q1/inv-1100.pdf", "count": 1}\n'
    '{"ts": "2026-03-13T14:46:23Z", "level": "INFO", "event": "document.view", "user": "acct.lead02", "src_ip": "10.10.40.63", "path": "/finance/2026-Q1/inv-1304.pdf", "count": 1}\n'
    '{"ts": "2026-03-13T14:46:23Z", "level": "INFO", "event": "document.view", "user": "acct.clerk01", "src_ip": "10.10.40.57", "path": "/finance/2026-Q1/inv-1683.pdf", "count": 1}\n'
    '{"ts": "2026-03-13T15:02:23Z", "level": "INFO", "event": "document.view", "user": "fin.mgr01", "src_ip": "10.10.40.63", "path": "/finance/2026-Q1/inv-1153.pdf", "count": 1}\n'
    '{"ts": "2026-03-13T15:36:23Z", "level": "INFO", "event": "document.view", "user": "fin.mgr01", "src_ip": "10.10.40.61", "path": "/finance/2026-Q1/inv-1823.pdf", "count": 1}\n'
    '{"ts": "2026-03-13T16:11:23Z", "level": "INFO", "event": "document.view", "user": "fin.mgr01", "src_ip": "10.10.40.61", "path": "/finance/2026-Q1/inv-1419.pdf", "count": 1}\n'
    '{"ts": "2026-03-13T16:16:23Z", "level": "INFO", "event": "document.view", "user": "fin.mgr01", "src_ip": "10.10.40.63", "path": "/finance/2026-Q1/inv-1024.pdf", "count": 1}\n'
    '{"ts": "2026-03-13T16:30:23Z", "level": "INFO", "event": "document.view", "user": "fin.mgr01", "src_ip": "10.10.40.63", "path": "/finance/2026-Q1/inv-1481.pdf", "count": 1}\n'
    '{"ts": "2026-03-13T16:39:23Z", "level": "INFO", "event": "document.view", "user": "acct.lead02", "src_ip": "10.10.40.63", "path": "/finance/2026-Q1/inv-1686.pdf", "count": 1}\n'
    '{"ts": "2026-03-13T16:47:23Z", "level": "INFO", "event": "document.view", "user": "fin.mgr01", "src_ip": "10.10.40.61", "path": "/finance/2026-Q1/inv-1461.pdf", "count": 1}\n'
    '{"ts": "2026-03-13T17:00:23Z", "level": "INFO", "event": "document.view", "user": "fin.mgr01", "src_ip": "10.10.40.61", "path": "/finance/2026-Q1/inv-1438.pdf", "count": 1}\n'
    '{"ts": "2026-03-14T03:24:33Z", "level": "INFO", "event": "auth.login", "user": "svc_backup", "src_ip": "127.0.0.1", "method": "local-token", "count": 1}\n'
    '{"ts": "2026-03-14T03:25:11Z", "level": "WARN", "event": "document.bulk_export", "user": "svc_backup", "src_ip": "127.0.0.1", "path": "/finance/2026-Q1", "count": 412, "bytes": 48006112}\n'
    '{"ts": "2026-03-14T03:25:25Z", "level": "INFO", "event": "export.complete", "user": "svc_backup", "path": "/srv/docportal/finance/2026-Q1", "count": 412}\n'
))
RESERVA['case-lab-p4/logs/firewall.log'] = ''.join((
    '2026-03-13 16:07:00 fw01 ACCEPT TCP 10.10.40.57:49982 -> 192.0.2.20:443 rule=allow-https-out bytes=45649\n'
    '2026-03-13 16:19:00 fw01 ACCEPT TCP 10.10.50.12:50042 -> 192.0.2.10:443 rule=allow-https-out bytes=11956\n'
    '2026-03-13 16:40:00 fw01 ACCEPT TCP 10.10.50.12:59993 -> 192.0.2.30:443 rule=allow-https-out bytes=11441\n'
    '2026-03-13 16:54:00 fw01 ACCEPT TCP 10.10.40.57:49616 -> 192.0.2.36:443 rule=allow-https-out bytes=78463\n'
    '2026-03-13 16:58:00 fw01 ACCEPT TCP 10.10.40.61:55186 -> 192.0.2.26:443 rule=allow-https-out bytes=3348\n'
    '2026-03-13 17:23:00 fw01 ACCEPT TCP 10.10.40.61:43475 -> 192.0.2.18:443 rule=allow-https-out bytes=28997\n'
    '2026-03-13 17:27:00 fw01 ACCEPT TCP 10.10.50.12:58821 -> 192.0.2.31:443 rule=allow-https-out bytes=20745\n'
    '2026-03-13 17:36:00 fw01 ACCEPT TCP 10.10.40.61:44782 -> 192.0.2.30:443 rule=allow-https-out bytes=74055\n'
    '2026-03-13 17:42:00 fw01 ACCEPT TCP 10.10.50.12:49971 -> 192.0.2.40:443 rule=allow-https-out bytes=2724\n'
    '2026-03-13 18:12:00 fw01 ACCEPT TCP 10.10.50.12:46834 -> 192.0.2.20:443 rule=allow-https-out bytes=37289\n'
    '2026-03-13 18:14:00 fw01 ACCEPT TCP 10.10.40.61:58905 -> 192.0.2.24:443 rule=allow-https-out bytes=24639\n'
    '2026-03-13 18:57:00 fw01 ACCEPT TCP 10.10.50.12:41396 -> 192.0.2.16:443 rule=allow-https-out bytes=24890\n'
    '2026-03-13 19:37:00 fw01 ACCEPT TCP 10.10.40.61:57120 -> 192.0.2.31:443 rule=allow-https-out bytes=81741\n'
    '2026-03-13 19:43:00 fw01 ACCEPT TCP 10.10.50.12:57897 -> 192.0.2.27:443 rule=allow-https-out bytes=3149\n'
    '2026-03-13 20:17:00 fw01 ACCEPT TCP 10.10.40.57:57888 -> 192.0.2.39:443 rule=allow-https-out bytes=28850\n'
    '2026-03-13 20:26:00 fw01 ACCEPT TCP 10.10.40.61:47688 -> 192.0.2.16:443 rule=allow-https-out bytes=45144\n'
    '2026-03-13 20:42:00 fw01 ACCEPT TCP 10.10.50.12:42196 -> 192.0.2.17:443 rule=allow-https-out bytes=70580\n'
    '2026-03-13 21:00:00 fw01 DROP TCP 198.51.100.71:56074 -> 10.10.20.5:3389 rule=default-deny bytes=60\n'
    '2026-03-13 21:02:00 fw01 DROP TCP 198.51.100.68:51554 -> 10.10.20.5:3389 rule=default-deny bytes=60\n'
    '2026-03-13 21:04:00 fw01 ACCEPT TCP 10.10.40.61:59948 -> 192.0.2.13:443 rule=allow-https-out bytes=41375\n'
    '2026-03-13 21:06:00 fw01 ACCEPT TCP 10.10.50.12:50162 -> 192.0.2.32:443 rule=allow-https-out bytes=37853\n'
    '2026-03-13 21:12:00 fw01 ACCEPT TCP 10.10.40.57:46775 -> 192.0.2.34:443 rule=allow-https-out bytes=20418\n'
    '2026-03-13 21:17:00 fw01 DROP TCP 198.51.100.86:49921 -> 10.10.20.5:3389 rule=default-deny bytes=60\n'
    '2026-03-13 21:27:00 fw01 DROP TCP 198.51.100.81:52296 -> 10.10.20.5:3389 rule=default-deny bytes=60\n'
    '2026-03-13 21:39:00 fw01 ACCEPT TCP 10.10.50.12:40560 -> 192.0.2.29:443 rule=allow-https-out bytes=44941\n'
    '2026-03-13 21:41:00 fw01 ACCEPT TCP 10.10.50.12:51843 -> 192.0.2.31:443 rule=allow-https-out bytes=64882\n'
    '2026-03-13 21:45:00 fw01 ACCEPT TCP 10.10.40.61:45200 -> 192.0.2.34:443 rule=allow-https-out bytes=5749\n'
    '2026-03-13 21:51:00 fw01 ACCEPT TCP 10.10.50.12:49874 -> 192.0.2.13:443 rule=allow-https-out bytes=23423\n'
    '2026-03-13 21:56:00 fw01 ACCEPT TCP 10.10.50.12:48605 -> 192.0.2.35:443 rule=allow-https-out bytes=18536\n'
    '2026-03-13 22:14:00 fw01 ACCEPT TCP 10.10.40.57:50184 -> 192.0.2.35:443 rule=allow-https-out bytes=65725\n'
    '2026-03-13 22:44:00 fw01 ACCEPT TCP 10.10.40.57:58676 -> 192.0.2.19:443 rule=allow-https-out bytes=21220\n'
    '2026-03-13 22:49:00 fw01 ACCEPT TCP 10.10.40.57:50528 -> 192.0.2.25:443 rule=allow-https-out bytes=23310\n'
    '2026-03-13 22:51:02 fw01 ACCEPT TCP 203.0.113.45:49208 -> 10.10.20.5:22 rule=allow-ssh-bastion bytes=4120\n'
    '2026-03-13 22:51:14 fw01 ACCEPT TCP 203.0.113.45:49212 -> 10.10.20.5:22 rule=allow-ssh-bastion bytes=4120\n'
    '2026-03-13 22:51:23 fw01 ACCEPT TCP 203.0.113.45:49216 -> 10.10.20.5:22 rule=allow-ssh-bastion bytes=4120\n'
    '2026-03-13 22:51:33 fw01 ACCEPT TCP 203.0.113.45:49224 -> 10.10.20.5:22 rule=allow-ssh-bastion bytes=4120\n'
    '2026-03-13 22:51:48 fw01 ACCEPT TCP 203.0.113.45:49230 -> 10.10.20.5:22 rule=allow-ssh-bastion bytes=4120\n'
    '2026-03-13 22:52:03 fw01 ACCEPT TCP 203.0.113.45:49237 -> 10.10.20.5:22 rule=allow-ssh-bastion bytes=4120\n'
    '2026-03-13 22:52:17 fw01 ACCEPT TCP 203.0.113.45:49245 -> 10.10.20.5:22 rule=allow-ssh-bastion bytes=4120\n'
    '2026-03-13 22:52:29 fw01 ACCEPT TCP 203.0.113.45:49249 -> 10.10.20.5:22 rule=allow-ssh-bastion bytes=4120\n'
    '2026-03-13 22:52:38 fw01 ACCEPT TCP 203.0.113.45:49252 -> 10.10.20.5:22 rule=allow-ssh-bastion bytes=4120\n'
    '2026-03-13 22:52:51 fw01 ACCEPT TCP 203.0.113.45:49256 -> 10.10.20.5:22 rule=allow-ssh-bastion bytes=4120\n'
    '2026-03-13 22:53:01 fw01 ACCEPT TCP 203.0.113.45:49264 -> 10.10.20.5:22 rule=allow-ssh-bastion bytes=4120\n'
    '2026-03-13 22:53:15 fw01 ACCEPT TCP 203.0.113.45:49268 -> 10.10.20.5:22 rule=allow-ssh-bastion bytes=4120\n'
    '2026-03-13 22:53:28 fw01 ACCEPT TCP 203.0.113.45:49269 -> 10.10.20.5:22 rule=allow-ssh-bastion bytes=4120\n'
    '2026-03-13 22:53:41 fw01 ACCEPT TCP 203.0.113.45:49278 -> 10.10.20.5:22 rule=allow-ssh-bastion bytes=4120\n'
    '2026-03-13 22:53:50 fw01 ACCEPT TCP 203.0.113.45:49279 -> 10.10.20.5:22 rule=allow-ssh-bastion bytes=4120\n'
    '2026-03-13 22:54:02 fw01 ACCEPT TCP 203.0.113.45:49287 -> 10.10.20.5:22 rule=allow-ssh-bastion bytes=4120\n'
    '2026-03-13 22:54:17 fw01 ACCEPT TCP 203.0.113.45:49296 -> 10.10.20.5:22 rule=allow-ssh-bastion bytes=4120\n'
    '2026-03-13 22:54:31 fw01 ACCEPT TCP 203.0.113.45:49303 -> 10.10.20.5:22 rule=allow-ssh-bastion bytes=4120\n'
    '2026-03-13 22:54:42 fw01 ACCEPT TCP 203.0.113.45:49311 -> 10.10.20.5:22 rule=allow-ssh-bastion bytes=4120\n'
    '2026-03-13 22:54:52 fw01 ACCEPT TCP 203.0.113.45:49316 -> 10.10.20.5:22 rule=allow-ssh-bastion bytes=4120\n'
    '2026-03-13 22:55:03 fw01 ACCEPT TCP 203.0.113.45:49321 -> 10.10.20.5:22 rule=allow-ssh-bastion bytes=4120\n'
    '2026-03-13 22:55:15 fw01 ACCEPT TCP 203.0.113.45:49330 -> 10.10.20.5:22 rule=allow-ssh-bastion bytes=4120\n'
    '2026-03-13 22:55:24 fw01 ACCEPT TCP 203.0.113.45:49337 -> 10.10.20.5:22 rule=allow-ssh-bastion bytes=4120\n'
    '2026-03-13 22:55:36 fw01 ACCEPT TCP 203.0.113.45:49343 -> 10.10.20.5:22 rule=allow-ssh-bastion bytes=4120\n'
    '2026-03-13 22:55:49 fw01 ACCEPT TCP 203.0.113.45:49347 -> 10.10.20.5:22 rule=allow-ssh-bastion bytes=4120\n'
    '2026-03-13 22:56:01 fw01 ACCEPT TCP 203.0.113.45:49348 -> 10.10.20.5:22 rule=allow-ssh-bastion bytes=4120\n'
    '2026-03-13 22:56:13 fw01 ACCEPT TCP 203.0.113.45:49357 -> 10.10.20.5:22 rule=allow-ssh-bastion bytes=4120\n'
    '2026-03-13 22:56:26 fw01 ACCEPT TCP 203.0.113.45:49359 -> 10.10.20.5:22 rule=allow-ssh-bastion bytes=4120\n'
    '2026-03-13 22:56:41 fw01 ACCEPT TCP 203.0.113.45:49368 -> 10.10.20.5:22 rule=allow-ssh-bastion bytes=4120\n'
    '2026-03-13 22:56:53 fw01 ACCEPT TCP 203.0.113.45:49374 -> 10.10.20.5:22 rule=allow-ssh-bastion bytes=4120\n'
    '2026-03-13 22:57:00 fw01 ACCEPT TCP 10.10.40.61:47756 -> 192.0.2.29:443 rule=allow-https-out bytes=85848\n'
    '2026-03-13 22:57:07 fw01 ACCEPT TCP 203.0.113.45:49376 -> 10.10.20.5:22 rule=allow-ssh-bastion bytes=4120\n'
    '2026-03-13 22:57:21 fw01 ACCEPT TCP 203.0.113.45:49379 -> 10.10.20.5:22 rule=allow-ssh-bastion bytes=4120\n'
    '2026-03-13 22:57:33 fw01 ACCEPT TCP 203.0.113.45:49383 -> 10.10.20.5:22 rule=allow-ssh-bastion bytes=4120\n'
    '2026-03-13 22:57:44 fw01 ACCEPT TCP 203.0.113.45:49387 -> 10.10.20.5:22 rule=allow-ssh-bastion bytes=4120\n'
    '2026-03-13 22:57:58 fw01 ACCEPT TCP 203.0.113.45:49395 -> 10.10.20.5:22 rule=allow-ssh-bastion bytes=4120\n'
    '2026-03-13 22:58:10 fw01 ACCEPT TCP 203.0.113.45:49398 -> 10.10.20.5:22 rule=allow-ssh-bastion bytes=4120\n'
    '2026-03-13 22:58:25 fw01 ACCEPT TCP 203.0.113.45:49403 -> 10.10.20.5:22 rule=allow-ssh-bastion bytes=4120\n'
    '2026-03-13 22:58:37 fw01 ACCEPT TCP 203.0.113.45:49410 -> 10.10.20.5:22 rule=allow-ssh-bastion bytes=4120\n'
    '2026-03-13 22:58:48 fw01 ACCEPT TCP 203.0.113.45:49411 -> 10.10.20.5:22 rule=allow-ssh-bastion bytes=4120\n'
    '2026-03-13 22:59:02 fw01 ACCEPT TCP 203.0.113.45:49419 -> 10.10.20.5:22 rule=allow-ssh-bastion bytes=4120\n'
    '2026-03-13 22:59:13 fw01 ACCEPT TCP 203.0.113.45:49422 -> 10.10.20.5:22 rule=allow-ssh-bastion bytes=4120\n'
    '2026-03-13 22:59:27 fw01 ACCEPT TCP 203.0.113.45:49430 -> 10.10.20.5:22 rule=allow-ssh-bastion bytes=4120\n'
    '2026-03-13 22:59:42 fw01 ACCEPT TCP 203.0.113.45:49434 -> 10.10.20.5:22 rule=allow-ssh-bastion bytes=4120\n'
    '2026-03-13 22:59:53 fw01 ACCEPT TCP 203.0.113.45:49436 -> 10.10.20.5:22 rule=allow-ssh-bastion bytes=4120\n'
    '2026-03-13 23:00:08 fw01 ACCEPT TCP 203.0.113.45:49440 -> 10.10.20.5:22 rule=allow-ssh-bastion bytes=4120\n'
    '2026-03-13 23:00:23 fw01 ACCEPT TCP 203.0.113.45:49443 -> 10.10.20.5:22 rule=allow-ssh-bastion bytes=4120\n'
    '2026-03-13 23:00:38 fw01 ACCEPT TCP 203.0.113.45:49447 -> 10.10.20.5:22 rule=allow-ssh-bastion bytes=4120\n'
    '2026-03-13 23:00:47 fw01 ACCEPT TCP 203.0.113.45:49450 -> 10.10.20.5:22 rule=allow-ssh-bastion bytes=4120\n'
    '2026-03-13 23:01:01 fw01 ACCEPT TCP 203.0.113.45:49458 -> 10.10.20.5:22 rule=allow-ssh-bastion bytes=4120\n'
    '2026-03-13 23:01:16 fw01 ACCEPT TCP 203.0.113.45:49461 -> 10.10.20.5:22 rule=allow-ssh-bastion bytes=4120\n'
    '2026-03-13 23:01:28 fw01 ACCEPT TCP 203.0.113.45:49470 -> 10.10.20.5:22 rule=allow-ssh-bastion bytes=4120\n'
    '2026-03-13 23:01:38 fw01 ACCEPT TCP 203.0.113.45:49478 -> 10.10.20.5:22 rule=allow-ssh-bastion bytes=4120\n'
    '2026-03-13 23:01:52 fw01 ACCEPT TCP 203.0.113.45:49486 -> 10.10.20.5:22 rule=allow-ssh-bastion bytes=4120\n'
    '2026-03-13 23:02:07 fw01 ACCEPT TCP 203.0.113.45:49491 -> 10.10.20.5:22 rule=allow-ssh-bastion bytes=4120\n'
    '2026-03-13 23:02:22 fw01 ACCEPT TCP 203.0.113.45:49499 -> 10.10.20.5:22 rule=allow-ssh-bastion bytes=4120\n'
    '2026-03-13 23:02:35 fw01 ACCEPT TCP 203.0.113.45:49504 -> 10.10.20.5:22 rule=allow-ssh-bastion bytes=4120\n'
    '2026-03-13 23:02:50 fw01 ACCEPT TCP 203.0.113.45:49511 -> 10.10.20.5:22 rule=allow-ssh-bastion bytes=4120\n'
    '2026-03-13 23:03:02 fw01 ACCEPT TCP 203.0.113.45:49512 -> 10.10.20.5:22 rule=allow-ssh-bastion bytes=4120\n'
    '2026-03-13 23:03:12 fw01 ACCEPT TCP 203.0.113.45:49517 -> 10.10.20.5:22 rule=allow-ssh-bastion bytes=4120\n'
    '2026-03-13 23:03:27 fw01 ACCEPT TCP 203.0.113.45:49524 -> 10.10.20.5:22 rule=allow-ssh-bastion bytes=4120\n'
    '2026-03-13 23:03:39 fw01 ACCEPT TCP 203.0.113.45:49525 -> 10.10.20.5:22 rule=allow-ssh-bastion bytes=4120\n'
    '2026-03-13 23:03:54 fw01 ACCEPT TCP 203.0.113.45:49534 -> 10.10.20.5:22 rule=allow-ssh-bastion bytes=4120\n'
    '2026-03-13 23:04:04 fw01 ACCEPT TCP 203.0.113.45:49539 -> 10.10.20.5:22 rule=allow-ssh-bastion bytes=4120\n'
    '2026-03-13 23:04:16 fw01 ACCEPT TCP 203.0.113.45:49547 -> 10.10.20.5:22 rule=allow-ssh-bastion bytes=4120\n'
    '2026-03-13 23:04:30 fw01 ACCEPT TCP 203.0.113.45:49556 -> 10.10.20.5:22 rule=allow-ssh-bastion bytes=4120\n'
    '2026-03-13 23:04:45 fw01 ACCEPT TCP 203.0.113.45:49561 -> 10.10.20.5:22 rule=allow-ssh-bastion bytes=4120\n'
    '2026-03-13 23:04:59 fw01 ACCEPT TCP 203.0.113.45:49565 -> 10.10.20.5:22 rule=allow-ssh-bastion bytes=4120\n'
    '2026-03-13 23:05:13 fw01 ACCEPT TCP 203.0.113.45:49568 -> 10.10.20.5:22 rule=allow-ssh-bastion bytes=4120\n'
    '2026-03-13 23:05:22 fw01 ACCEPT TCP 203.0.113.45:49576 -> 10.10.20.5:22 rule=allow-ssh-bastion bytes=4120\n'
    '2026-03-13 23:05:31 fw01 ACCEPT TCP 203.0.113.45:49577 -> 10.10.20.5:22 rule=allow-ssh-bastion bytes=4120\n'
    '2026-03-13 23:05:43 fw01 ACCEPT TCP 203.0.113.45:49582 -> 10.10.20.5:22 rule=allow-ssh-bastion bytes=4120\n'
    '2026-03-13 23:05:54 fw01 ACCEPT TCP 203.0.113.45:49589 -> 10.10.20.5:22 rule=allow-ssh-bastion bytes=4120\n'
    '2026-03-13 23:06:05 fw01 ACCEPT TCP 203.0.113.45:49593 -> 10.10.20.5:22 rule=allow-ssh-bastion bytes=4120\n'
    '2026-03-13 23:06:20 fw01 ACCEPT TCP 203.0.113.45:49598 -> 10.10.20.5:22 rule=allow-ssh-bastion bytes=4120\n'
    '2026-03-13 23:06:33 fw01 ACCEPT TCP 203.0.113.45:49602 -> 10.10.20.5:22 rule=allow-ssh-bastion bytes=4120\n'
    '2026-03-13 23:06:47 fw01 ACCEPT TCP 203.0.113.45:49605 -> 10.10.20.5:22 rule=allow-ssh-bastion bytes=4120\n'
    '2026-03-13 23:06:56 fw01 ACCEPT TCP 203.0.113.45:49614 -> 10.10.20.5:22 rule=allow-ssh-bastion bytes=4120\n'
    '2026-03-13 23:07:06 fw01 ACCEPT TCP 203.0.113.45:49617 -> 10.10.20.5:22 rule=allow-ssh-bastion bytes=4120\n'
    '2026-03-13 23:07:17 fw01 ACCEPT TCP 203.0.113.45:49625 -> 10.10.20.5:22 rule=allow-ssh-bastion bytes=4120\n'
    '2026-03-13 23:07:29 fw01 ACCEPT TCP 203.0.113.45:49633 -> 10.10.20.5:22 rule=allow-ssh-bastion bytes=4120\n'
    '2026-03-13 23:07:38 fw01 ACCEPT TCP 203.0.113.45:49636 -> 10.10.20.5:22 rule=allow-ssh-bastion bytes=4120\n'
    '2026-03-13 23:07:50 fw01 ACCEPT TCP 203.0.113.45:49640 -> 10.10.20.5:22 rule=allow-ssh-bastion bytes=4120\n'
    '2026-03-13 23:08:00 fw01 ACCEPT TCP 203.0.113.45:49647 -> 10.10.20.5:22 rule=allow-ssh-bastion bytes=4120\n'
    '2026-03-13 23:08:09 fw01 ACCEPT TCP 203.0.113.45:49649 -> 10.10.20.5:22 rule=allow-ssh-bastion bytes=4120\n'
    '2026-03-13 23:08:18 fw01 ACCEPT TCP 203.0.113.45:49651 -> 10.10.20.5:22 rule=allow-ssh-bastion bytes=4120\n'
    '2026-03-13 23:08:27 fw01 ACCEPT TCP 203.0.113.45:49655 -> 10.10.20.5:22 rule=allow-ssh-bastion bytes=4120\n'
    '2026-03-13 23:08:41 fw01 ACCEPT TCP 203.0.113.45:49657 -> 10.10.20.5:22 rule=allow-ssh-bastion bytes=4120\n'
    '2026-03-13 23:08:53 fw01 ACCEPT TCP 203.0.113.45:49658 -> 10.10.20.5:22 rule=allow-ssh-bastion bytes=4120\n'
    '2026-03-13 23:09:02 fw01 ACCEPT TCP 203.0.113.45:49659 -> 10.10.20.5:22 rule=allow-ssh-bastion bytes=4120\n'
    '2026-03-13 23:09:13 fw01 ACCEPT TCP 203.0.113.45:49664 -> 10.10.20.5:22 rule=allow-ssh-bastion bytes=4120\n'
    '2026-03-13 23:09:24 fw01 ACCEPT TCP 203.0.113.45:49665 -> 10.10.20.5:22 rule=allow-ssh-bastion bytes=4120\n'
    '2026-03-13 23:09:36 fw01 ACCEPT TCP 203.0.113.45:49673 -> 10.10.20.5:22 rule=allow-ssh-bastion bytes=4120\n'
    '2026-03-13 23:09:50 fw01 ACCEPT TCP 203.0.113.45:49681 -> 10.10.20.5:22 rule=allow-ssh-bastion bytes=4120\n'
    '2026-03-13 23:09:59 fw01 ACCEPT TCP 203.0.113.45:49689 -> 10.10.20.5:22 rule=allow-ssh-bastion bytes=4120\n'
    '2026-03-13 23:10:09 fw01 ACCEPT TCP 203.0.113.45:49696 -> 10.10.20.5:22 rule=allow-ssh-bastion bytes=4120\n'
    '2026-03-13 23:10:18 fw01 ACCEPT TCP 203.0.113.45:49703 -> 10.10.20.5:22 rule=allow-ssh-bastion bytes=4120\n'
    '2026-03-13 23:10:32 fw01 ACCEPT TCP 203.0.113.45:49712 -> 10.10.20.5:22 rule=allow-ssh-bastion bytes=4120\n'
    '2026-03-13 23:10:46 fw01 ACCEPT TCP 203.0.113.45:49713 -> 10.10.20.5:22 rule=allow-ssh-bastion bytes=4120\n'
    '2026-03-13 23:11:01 fw01 ACCEPT TCP 203.0.113.45:49720 -> 10.10.20.5:22 rule=allow-ssh-bastion bytes=4120\n'
    '2026-03-13 23:11:11 fw01 ACCEPT TCP 203.0.113.45:49727 -> 10.10.20.5:22 rule=allow-ssh-bastion bytes=4120\n'
    '2026-03-13 23:11:21 fw01 ACCEPT TCP 203.0.113.45:49734 -> 10.10.20.5:22 rule=allow-ssh-bastion bytes=4120\n'
    '2026-03-13 23:11:35 fw01 ACCEPT TCP 203.0.113.45:49742 -> 10.10.20.5:22 rule=allow-ssh-bastion bytes=4120\n'
    '2026-03-13 23:11:48 fw01 ACCEPT TCP 203.0.113.45:49744 -> 10.10.20.5:22 rule=allow-ssh-bastion bytes=4120\n'
    '2026-03-13 23:12:03 fw01 ACCEPT TCP 203.0.113.45:49745 -> 10.10.20.5:22 rule=allow-ssh-bastion bytes=4120\n'
    '2026-03-13 23:12:12 fw01 ACCEPT TCP 203.0.113.45:49751 -> 10.10.20.5:22 rule=allow-ssh-bastion bytes=4120\n'
    '2026-03-13 23:12:22 fw01 ACCEPT TCP 203.0.113.45:49753 -> 10.10.20.5:22 rule=allow-ssh-bastion bytes=4120\n'
    '2026-03-13 23:12:36 fw01 ACCEPT TCP 203.0.113.45:49755 -> 10.10.20.5:22 rule=allow-ssh-bastion bytes=4120\n'
    '2026-03-13 23:12:46 fw01 ACCEPT TCP 203.0.113.45:49759 -> 10.10.20.5:22 rule=allow-ssh-bastion bytes=4120\n'
    '2026-03-13 23:13:00 fw01 ACCEPT TCP 203.0.113.45:49760 -> 10.10.20.5:22 rule=allow-ssh-bastion bytes=4120\n'
    '2026-03-13 23:13:14 fw01 ACCEPT TCP 203.0.113.45:49766 -> 10.10.20.5:22 rule=allow-ssh-bastion bytes=4120\n'
    '2026-03-13 23:13:23 fw01 ACCEPT TCP 203.0.113.45:49774 -> 10.10.20.5:22 rule=allow-ssh-bastion bytes=4120\n'
    '2026-03-13 23:13:35 fw01 ACCEPT TCP 203.0.113.45:49783 -> 10.10.20.5:22 rule=allow-ssh-bastion bytes=4120\n'
    '2026-03-13 23:14:06 fw01 ACCEPT TCP 203.0.113.45:50122 -> 10.10.20.5:22 rule=allow-ssh-bastion bytes=4120\n'
    '2026-03-13 23:19:21 fw01 ACCEPT TCP 10.10.20.5:41766 -> 10.10.30.17:22 rule=allow-ssh-internal bytes=18844\n'
    '2026-03-13 23:40:00 fw01 DROP TCP 198.51.100.68:52943 -> 10.10.20.5:3389 rule=default-deny bytes=60\n'
    '2026-03-13 23:41:00 fw01 ACCEPT UDP 10.10.30.17:53422 -> 10.10.0.2:53 rule=allow-dns-internal bytes=92\n'
    '2026-03-13 23:41:02 fw01 ACCEPT TCP 10.10.30.17:39514 -> 198.51.100.23:443 rule=allow-https-out bytes=48213904\n'
    '2026-03-13 23:50:00 fw01 DROP TCP 198.51.100.90:50796 -> 10.10.20.5:3389 rule=default-deny bytes=60\n'
    '2026-03-13 23:55:00 fw01 ACCEPT TCP 10.10.50.12:45889 -> 192.0.2.19:443 rule=allow-https-out bytes=35329\n'
))
RESERVA['case-lab-p4/logs/fs01-auth.log'] = ''.join((
    'Mar 13 09:16:25 fs01 sshd[1188]: Accepted publickey for ops.admin from 10.10.50.12 port 50912 ssh2: ED25519 SHA256:q8m1Xx0lab0synthetic0key0fingerprint0AAAA\n'
    'Mar 14 03:20:45 fs01 sshd[30411]: Accepted publickey for svc_backup from 10.10.20.5 port 41766 ssh2: RSA SHA256:Zb7synthetic0lab0svcbackup0key0BBBBBBBBBB\n'
    'Mar 14 03:20:46 fs01 sshd[30411]: pam_unix(sshd:session): session opened for user svc_backup(uid=1107) by (uid=0)\n'
    'Mar 14 03:33:03 fs01 sudo[30502]: svc_backup : TTY=pts/1 ; PWD=/home/svc_backup ; USER=root ; COMMAND=/usr/bin/tar -czf /var/tmp/.q1.tgz /srv/docportal/finance/2026-Q1\n'
    'Mar 14 03:33:03 fs01 sudo[30502]: pam_unix(sudo:session): session opened for user root(uid=0) by svc_backup(uid=1107)\n'
    'Mar 14 03:34:28 fs01 sudo[30502]: pam_unix(sudo:session): session closed for user root\n'
    'Mar 14 03:51:21 fs01 sshd[30411]: pam_unix(sshd:session): session closed for user svc_backup\n'
))

EVIDENCIA = {}           # o que você vai analisar de fato (nome curto -> texto)
ORIGEM = {}              # de onde veio cada arquivo: "rede" ou "reserva"

for nome, reserva in RESERVA.items():
    texto = None
    if not FORCAR_OFFLINE:
        try:
            # timeout curto: se a rede não responder em 8 s, seguimos com o plano B
            with urllib.request.urlopen(URL_BASE + nome, timeout=8) as resp:
                texto = resp.read().decode("utf-8")
            ORIGEM[nome] = "rede"
        except Exception as erro:            # sem internet, proxy, 404... qualquer falha
            print(f"⚠️  {nome.split('/')[-1]}: rede indisponível ({type(erro).__name__}), usando a cópia embutida")
    if texto is None:
        texto = reserva
        ORIGEM[nome] = "reserva"
    EVIDENCIA[nome.split("/")[-1]] = texto   # guardamos pelo nome curto
    h = hashlib.sha256(texto.encode("utf-8")).hexdigest()
    # Se o upstream mudou depois que este notebook foi gerado, os hashes divergem: avisamos.
    igual = "= cópia embutida" if texto == reserva else "≠ cópia embutida (o upstream mudou!)"
    print(f"{ORIGEM[nome]:8} {h[:16]}…  {nome.split('/')[-1]}  ({igual})")


## 2. Original separado da cópia

O original diz `mkdir -p ~/lab-p4/{evidence,work,notes}`, `cp -a` e `chmod -R a-w`. Aqui fazemos o mesmo em Python, numa pasta `lab-p4/` ao lado do notebook.


In [ ]:
import os, stat, shutil                     # arquivos, permissões e cópias
from pathlib import Path
from datetime import datetime, timezone

BASE = Path("lab-p4")
for sub in ("evidence", "work", "notes"):     # o mesmo que mkdir -p ~/lab-p4/{evidence,work,notes}
    (BASE / sub).mkdir(parents=True, exist_ok=True)

ORIGINAL = BASE / "evidence" / "EVID-001"
if ORIGINAL.exists():                         # se você rodar de novo, devolvemos a escrita para poder recriar
    for p in ORIGINAL.iterdir():
        p.chmod(stat.S_IRUSR | stat.S_IWUSR)
    ORIGINAL.chmod(0o755)
    shutil.rmtree(ORIGINAL)
ORIGINAL.mkdir()

for nome, texto in EVIDENCIA.items():
    # newline="" evita que o Windows troque \n por \r\n: um byte a mais e o hash muda!
    with open(ORIGINAL / nome, "w", encoding="utf-8", newline="") as f:
        f.write(texto)

def so_leitura(pasta):
    """O nosso 'chmod -R a-w': tira a escrita de todos (substituto de laboratório para o write blocker)."""
    for p in pasta.iterdir():
        p.chmod(stat.S_IRUSR | stat.S_IRGRP | stat.S_IROTH)

so_leitura(ORIGINAL)

def registrar_acao(texto, log=BASE / "notes" / "actions.log"):
    """Uma linha por ação, com hora UTC, como o `date -u ... >> actions.log` do original."""
    agora = datetime.now(timezone.utc).strftime("%Y-%m-%dT%H:%M:%SZ")
    with open(log, "a", encoding="utf-8") as f:
        f.write(f"{agora} {texto}\n")
    print(f"📝 {agora} {texto}")

for p in sorted(ORIGINAL.iterdir()):
    print(f"{stat.filemode(p.stat().st_mode)}  {p.stat().st_size:>7} bytes  {p.name}")


## 3. O manifesto do recebimento

`sha256sum *` imprime `hash`, **dois espaços**, `nome`. Copiamos esse formato à risca, porque assim o seu manifesto também funciona com o `sha256sum -c` de verdade num Linux.


In [ ]:
import hashlib

def sha256_arquivo(caminho):
    """Lê em blocos de 1 MiB: funciona igual para um log de 1 KB e uma imagem de 500 GB."""
    h = hashlib.sha256()
    with open(caminho, "rb") as f:                 # "rb": bytes crus, sem traduzir nada
        for bloco in iter(lambda: f.read(1 << 20), b""):
            h.update(bloco)
    return h.hexdigest()

def gerar_manifesto(pasta, destino):
    """Equivale a: cd pasta && sha256sum * | tee destino"""
    linhas = [f"{sha256_arquivo(p)}  {p.name}" for p in sorted(pasta.iterdir()) if p.is_file()]
    Path(destino).write_text("\n".join(linhas) + "\n", encoding="utf-8")
    print("\n".join(linhas))
    return destino

MANIFESTO = gerar_manifesto(ORIGINAL, BASE / "notes" / "EVID-001.receipt.sha256")
registrar_acao("received EVID-001")


## 4. Cópia de trabalho e `sha256sum -c`

Agora a cópia. E a prova de que ela é idêntica: recalcular cada hash e comparar com o manifesto.


In [ ]:
COPIA = BASE / "work" / "EVID-001"

def fazer_copia():
    """O nosso 'cp -a': copia preservando os tempos (copy2) e devolve escrita à CÓPIA."""
    if COPIA.exists():
        shutil.rmtree(COPIA)
    shutil.copytree(ORIGINAL, COPIA, copy_function=shutil.copy2)
    for p in COPIA.iterdir():
        p.chmod(stat.S_IRUSR | stat.S_IWUSR | stat.S_IRGRP | stat.S_IROTH)

def conferir(pasta, manifesto):
    """Equivale a sha256sum -c: devolve quantos arquivos falharam."""
    falhas = 0
    for linha in Path(manifesto).read_text(encoding="utf-8").splitlines():
        esperado, nome = linha.split("  ", 1)       # separa no DOIS espaços
        alvo = pasta / nome
        ok = alvo.exists() and sha256_arquivo(alvo) == esperado
        print(f"{nome}: {'OK' if ok else 'FAILED'}")
        falhas += (not ok)
    if falhas:
        print(f"sha256sum: WARNING: {falhas} computed checksum{'s' if falhas > 1 else ''} did NOT match")
    return falhas

fazer_copia()
registrar_acao("working copy work/EVID-001 created from evidence/EVID-001")
assert conferir(COPIA, MANIFESTO) == 0, "A cópia não confere: não analise nada!"


## 5. 💥 O teste de adulteração

Um espaço no fim do `firewall.log`. Só isso.


In [ ]:
with open(COPIA / "firewall.log", "a", encoding="utf-8") as f:   # printf ' ' >> firewall.log
    f.write(" ")

falhas = conferir(COPIA, MANIFESTO)
registrar_acao(f"TAMPER TEST: appended one space to work/EVID-001/firewall.log; sha256sum -c -> {falhas} FAILED")

# O procedimento correto: apagar a cópia e refazer a partir do original (nunca "consertar" a cópia).
fazer_copia()
registrar_acao("working copy deleted and replaced from evidence/EVID-001")
print("\nDepois de refazer a cópia:")
assert conferir(COPIA, MANIFESTO) == 0


> 💡 **Repare** no efeito avalanche: um byte a mais muda o hash inteiro, não "um pouquinho". Por isso o hash prova *integridade*, mas não diz *o que* mudou. Para isso você precisaria comparar o conteúdo (o `diff`).

## 6. EVID-002 e EVID-003

O passo 6 do original: manifestos para as pastas Windows + sistema de arquivos (EVID-002) e memória (EVID-003).


In [ ]:
# ------------------------------------------------------------
# RECEBER O EVID-002 E O EVID-003
# Plano A: baixar do repositório original (GitHub raw), com urllib.
# Plano B: usar a cópia embutida logo abaixo (o arquivo inteiro,
#          copiado do upstream/), para o notebook rodar sem internet.
# ------------------------------------------------------------
import urllib.request      # baixa arquivos da web, sem instalar nada
import hashlib             # SHA-256: a impressão digital dos bytes

FORCAR_OFFLINE = False     # True = nem tenta a rede (útil em sala sem Wi-Fi)
URL_BASE = 'https://raw.githubusercontent.com/thiefdroppers/cyber-investigator-skills/main/curriculum/phase-4-digital-forensics-incident-investigation/resources/'

RESERVA = {}             # caminho do arquivo -> conteúdo embutido (plano B)
RESERVA['case-lab-p4/windows/ws-fin-07-security.csv'] = ''.join((
    'TimeCreatedUtc,EventID,TargetUserName,LogonType,IpAddress,WorkstationName\n'
    '2026-03-09T05:24:00Z,4624,SYSTEM,5,-,WS-FIN-07\n'
    '2026-03-09T10:14:00Z,4624,SYSTEM,5,-,WS-FIN-07\n'
    '2026-03-09T13:03:23Z,4624,acct.clerk01,7,-,WS-FIN-07\n'
    '2026-03-09T13:13:07Z,4624,acct.clerk01,11,-,WS-FIN-07\n'
    '2026-03-09T13:51:21Z,4624,acct.clerk01,7,-,WS-FIN-07\n'
    '2026-03-09T14:20:23Z,4624,acct.clerk01,2,-,WS-FIN-07\n'
    '2026-03-09T14:29:00Z,4625,acct.clerk01,2,-,WS-FIN-07\n'
    '2026-03-09T15:08:51Z,4624,acct.clerk01,7,-,WS-FIN-07\n'
    '2026-03-09T15:47:41Z,4624,acct.clerk01,2,-,WS-FIN-07\n'
    '2026-03-09T16:20:19Z,4624,acct.clerk01,11,-,WS-FIN-07\n'
    '2026-03-09T16:49:36Z,4624,acct.lead02,2,-,WS-FIN-07\n'
    '2026-03-09T17:04:59Z,4624,acct.clerk01,2,-,WS-FIN-07\n'
    '2026-03-09T17:36:39Z,4624,acct.clerk01,11,-,WS-FIN-07\n'
    '2026-03-09T17:48:37Z,4624,acct.clerk01,7,-,WS-FIN-07\n'
    '2026-03-09T18:34:14Z,4624,acct.lead02,7,-,WS-FIN-07\n'
    '2026-03-09T19:09:22Z,4624,acct.lead02,2,-,WS-FIN-07\n'
    '2026-03-09T19:50:54Z,4624,acct.clerk01,2,-,WS-FIN-07\n'
    '2026-03-09T21:20:56Z,4624,acct.clerk01,11,-,WS-FIN-07\n'
    '2026-03-09T21:39:22Z,4624,acct.clerk01,2,-,WS-FIN-07\n'
    '2026-03-09T21:49:17Z,4624,acct.clerk01,2,-,WS-FIN-07\n'
    '2026-03-10T06:42:00Z,4624,SYSTEM,5,-,WS-FIN-07\n'
    '2026-03-10T14:31:10Z,4624,acct.clerk01,11,-,WS-FIN-07\n'
    '2026-03-10T15:00:07Z,4624,acct.clerk01,7,-,WS-FIN-07\n'
    '2026-03-10T15:08:10Z,4624,acct.lead02,7,-,WS-FIN-07\n'
    '2026-03-10T15:23:26Z,4624,acct.clerk01,7,-,WS-FIN-07\n'
    '2026-03-10T15:44:47Z,4624,acct.clerk01,2,-,WS-FIN-07\n'
    '2026-03-10T16:08:00Z,4625,acct.clerk01,2,-,WS-FIN-07\n'
    '2026-03-10T16:21:17Z,4624,acct.clerk01,2,-,WS-FIN-07\n'
    '2026-03-10T16:36:00Z,4624,SYSTEM,5,-,WS-FIN-07\n'
    '2026-03-10T16:55:35Z,4624,acct.clerk01,11,-,WS-FIN-07\n'
    '2026-03-10T17:04:04Z,4624,acct.clerk01,7,-,WS-FIN-07\n'
    '2026-03-10T17:25:00Z,4624,acct.lead02,2,-,WS-FIN-07\n'
    '2026-03-10T19:19:06Z,4624,acct.lead02,2,-,WS-FIN-07\n'
    '2026-03-10T19:41:08Z,4624,acct.clerk01,11,-,WS-FIN-07\n'
    '2026-03-10T20:06:33Z,4624,acct.clerk01,7,-,WS-FIN-07\n'
    '2026-03-10T20:48:43Z,4624,acct.clerk01,11,-,WS-FIN-07\n'
    '2026-03-11T12:35:00Z,4624,SYSTEM,5,-,WS-FIN-07\n'
    '2026-03-11T13:08:15Z,4624,acct.lead02,7,-,WS-FIN-07\n'
    '2026-03-11T13:16:22Z,4624,acct.clerk01,11,-,WS-FIN-07\n'
    '2026-03-11T13:17:29Z,4624,acct.lead02,11,-,WS-FIN-07\n'
    '2026-03-11T13:18:42Z,4624,acct.clerk01,7,-,WS-FIN-07\n'
    '2026-03-11T14:25:44Z,4624,acct.lead02,11,-,WS-FIN-07\n'
    '2026-03-11T14:50:25Z,4624,acct.clerk01,2,-,WS-FIN-07\n'
    '2026-03-11T15:27:00Z,4625,acct.clerk01,2,-,WS-FIN-07\n'
    '2026-03-11T15:43:00Z,4624,SYSTEM,5,-,WS-FIN-07\n'
    '2026-03-11T16:33:55Z,4624,acct.clerk01,7,-,WS-FIN-07\n'
    '2026-03-11T16:39:08Z,4624,acct.clerk01,7,-,WS-FIN-07\n'
    '2026-03-11T16:48:16Z,4624,acct.clerk01,7,-,WS-FIN-07\n'
    '2026-03-11T17:00:12Z,4624,acct.clerk01,11,-,WS-FIN-07\n'
    '2026-03-11T17:35:19Z,4624,acct.clerk01,7,-,WS-FIN-07\n'
    '2026-03-11T17:55:17Z,4624,acct.lead02,11,-,WS-FIN-07\n'
    '2026-03-11T18:41:35Z,4624,acct.clerk01,11,-,WS-FIN-07\n'
    '2026-03-11T19:08:40Z,4624,acct.clerk01,7,-,WS-FIN-07\n'
    '2026-03-11T19:09:22Z,4624,acct.lead02,7,-,WS-FIN-07\n'
    '2026-03-11T19:10:57Z,4624,acct.clerk01,7,-,WS-FIN-07\n'
    '2026-03-11T19:21:53Z,4624,acct.clerk01,7,-,WS-FIN-07\n'
    '2026-03-11T19:25:43Z,4624,acct.lead02,11,-,WS-FIN-07\n'
    '2026-03-11T20:59:45Z,4624,acct.clerk01,11,-,WS-FIN-07\n'
    '2026-03-12T05:04:00Z,4624,SYSTEM,5,-,WS-FIN-07\n'
    '2026-03-12T06:39:00Z,4624,SYSTEM,5,-,WS-FIN-07\n'
    '2026-03-12T14:13:21Z,4624,acct.clerk01,2,-,WS-FIN-07\n'
    '2026-03-12T14:25:26Z,4624,acct.clerk01,2,-,WS-FIN-07\n'
    '2026-03-12T14:55:50Z,4624,acct.clerk01,11,-,WS-FIN-07\n'
    '2026-03-12T15:01:16Z,4624,acct.clerk01,11,-,WS-FIN-07\n'
    '2026-03-12T15:11:04Z,4624,acct.clerk01,7,-,WS-FIN-07\n'
    '2026-03-12T15:27:43Z,4624,acct.lead02,7,-,WS-FIN-07\n'
    '2026-03-12T16:57:07Z,4624,acct.lead02,2,-,WS-FIN-07\n'
    '2026-03-12T18:12:40Z,4624,acct.clerk01,7,-,WS-FIN-07\n'
    '2026-03-12T18:28:34Z,4624,acct.clerk01,11,-,WS-FIN-07\n'
    '2026-03-12T18:33:25Z,4624,acct.clerk01,11,-,WS-FIN-07\n'
    '2026-03-12T18:43:05Z,4624,acct.lead02,2,-,WS-FIN-07\n'
    '2026-03-12T19:39:27Z,4624,acct.clerk01,11,-,WS-FIN-07\n'
    '2026-03-12T19:44:51Z,4624,acct.clerk01,7,-,WS-FIN-07\n'
    '2026-03-12T19:46:00Z,4625,acct.clerk01,2,-,WS-FIN-07\n'
    '2026-03-12T20:20:51Z,4624,acct.clerk01,7,-,WS-FIN-07\n'
    '2026-03-12T21:01:20Z,4624,acct.lead02,2,-,WS-FIN-07\n'
    '2026-03-13T13:17:00Z,4624,SYSTEM,5,-,WS-FIN-07\n'
    '2026-03-13T13:27:56Z,4624,acct.clerk01,11,-,WS-FIN-07\n'
    '2026-03-13T15:24:20Z,4624,acct.clerk01,2,-,WS-FIN-07\n'
    '2026-03-13T16:25:00Z,4624,SYSTEM,5,-,WS-FIN-07\n'
    '2026-03-13T17:04:28Z,4624,acct.clerk01,7,-,WS-FIN-07\n'
    '2026-03-13T17:13:10Z,4624,acct.clerk01,7,-,WS-FIN-07\n'
    '2026-03-13T17:45:27Z,4624,acct.clerk01,2,-,WS-FIN-07\n'
    '2026-03-13T18:45:06Z,4624,acct.clerk01,2,-,WS-FIN-07\n'
    '2026-03-13T19:13:05Z,4624,acct.clerk01,2,-,WS-FIN-07\n'
    '2026-03-13T19:28:00Z,4625,acct.clerk01,2,-,WS-FIN-07\n'
    '2026-03-13T19:31:08Z,4624,acct.lead02,2,-,WS-FIN-07\n'
    '2026-03-13T19:34:17Z,4624,acct.clerk01,7,-,WS-FIN-07\n'
    '2026-03-13T20:17:59Z,4624,acct.clerk01,11,-,WS-FIN-07\n'
    '2026-03-13T20:32:28Z,4624,acct.clerk01,7,-,WS-FIN-07\n'
    '2026-03-13T21:37:18Z,4624,acct.clerk01,11,-,WS-FIN-07\n'
    '2026-03-13T21:48:40Z,4624,acct.lead02,11,-,WS-FIN-07\n'
    '2026-03-13T21:50:09Z,4624,acct.lead02,2,-,WS-FIN-07\n'
    '2026-03-13T21:59:56Z,4624,acct.clerk01,11,-,WS-FIN-07\n'
    '2026-03-14T02:10:00Z,4625,administrator,3,10.10.20.5,-\n'
    '2026-03-14T02:10:17Z,4625,administrator,3,10.10.20.5,-\n'
    '2026-03-14T02:10:34Z,4625,svc_backup,3,10.10.20.5,-\n'
    '2026-03-14T02:10:51Z,4625,administrator,3,10.10.20.5,-\n'
    '2026-03-14T02:11:08Z,4625,acct.clerk01,3,10.10.20.5,-\n'
    '2026-03-14T02:11:25Z,4625,acct.clerk01,3,10.10.20.5,-\n'
    '2026-03-14T02:11:42Z,4625,administrator,3,10.10.20.5,-\n'
    '2026-03-14T02:11:59Z,4625,acct.clerk01,3,10.10.20.5,-\n'
    '2026-03-14T02:12:16Z,4625,acct.clerk01,3,10.10.20.5,-\n'
    '2026-03-14T02:12:33Z,4625,administrator,3,10.10.20.5,-\n'
    '2026-03-14T02:12:50Z,4625,svc_backup,3,10.10.20.5,-\n'
    '2026-03-14T02:13:07Z,4625,acct.clerk01,3,10.10.20.5,-\n'
    '2026-03-14T02:13:24Z,4625,administrator,3,10.10.20.5,-\n'
    '2026-03-14T02:13:41Z,4625,acct.clerk01,3,10.10.20.5,-\n'
    '2026-03-14T02:13:58Z,4625,administrator,3,10.10.20.5,-\n'
    '2026-03-14T02:14:15Z,4625,administrator,3,10.10.20.5,-\n'
    '2026-03-14T02:14:32Z,4625,administrator,3,10.10.20.5,-\n'
    '2026-03-14T02:14:49Z,4625,administrator,3,10.10.20.5,-\n'
    '2026-03-14T02:15:06Z,4625,svc_backup,3,10.10.20.5,-\n'
    '2026-03-14T02:15:23Z,4625,administrator,3,10.10.20.5,-\n'
    '2026-03-14T02:15:40Z,4625,administrator,3,10.10.20.5,-\n'
    '2026-03-14T02:15:57Z,4625,administrator,3,10.10.20.5,-\n'
    '2026-03-14T02:16:14Z,4625,administrator,3,10.10.20.5,-\n'
    '2026-03-14T03:27:41Z,4624,svc_backup,3,10.10.30.17,-\n'
    '2026-03-14T03:27:41Z,4672,svc_backup,,,\n'
    '2026-03-14T06:43:00Z,4624,SYSTEM,5,-,WS-FIN-07\n'
    '2026-03-14T12:50:00Z,4624,SYSTEM,5,-,WS-FIN-07\n'
))
RESERVA['case-lab-p4/filesystem/ws-fin-07-mft-excerpt.csv'] = ''.join((
    'EntryNumber,SequenceNumber,InUse,ParentEntryNumber,ParentPath,FileName,Extension,FileSize,IsDirectory,SI<FN,uSecZeros,Created0x10,Created0x30,LastModified0x10,LastModified0x30,LastRecordChange0x10,LastRecordChange0x30,LastAccess0x10,LastAccess0x30\n'
    '88412,3,True,88390,.\\Users\\acct.clerk01\\AppData\\Roaming,SyncHelper,,0,True,False,False,2026-03-13 16:42:29.5528113,2026-03-13 16:42:29.5528113,2026-03-13 16:42:30.1102964,2026-03-13 16:42:29.5528113,2026-03-13 16:42:30.1102964,2026-03-13 16:42:29.5528113,2026-03-13 16:42:30.1102964,2026-03-13 16:42:29.5528113\n'
    '88413,1,True,88412,.\\Users\\acct.clerk01\\AppData\\Roaming\\SyncHelper,synchelper.exe,.exe,412160,False,True,True,2019-06-11 09:14:00.0000000,2026-03-13 16:42:30.1102964,2019-06-11 09:14:00.0000000,2026-03-13 16:42:30.1102964,2026-03-13 16:42:31.0470021,2026-03-13 16:42:30.1102964,2026-03-13 16:42:31.0470021,2026-03-13 16:42:30.1102964\n'
    '88414,1,True,88412,.\\Users\\acct.clerk01\\AppData\\Roaming\\SyncHelper,config.dat,.dat,1896,False,False,False,2026-03-13 16:42:30.2231870,2026-03-13 16:42:30.2231870,2026-03-13 16:42:31.3009144,2026-03-13 16:42:30.2231870,2026-03-13 16:42:31.3009144,2026-03-13 16:42:30.2231870,2026-03-13 16:42:31.3009144,2026-03-13 16:42:30.2231870\n'
    '88420,2,True,1471,.\\Users\\Public,svchost.exe,.exe,98304,False,True,True,2018-09-15 07:28:00.0000000,2026-03-13 16:44:08.8811220,2018-09-15 07:28:00.0000000,2026-03-13 16:44:08.8811220,2026-03-13 16:44:09.0034187,2026-03-13 16:44:08.8811220,2026-03-13 16:44:09.0034187,2026-03-13 16:44:08.8811220\n'
    '23105,7,True,23100,.\\Windows\\System32,svchost.exe,.exe,57528,False,False,False,2025-11-12 03:41:17.4406211,2025-11-12 03:41:17.4406211,2025-11-12 03:41:17.4562810,2025-11-12 03:41:17.4406211,2025-11-12 03:41:17.4562810,2025-11-12 03:41:17.4406211,2026-03-13 12:58:04.1101114,2025-11-12 03:41:17.4406211\n'
    '77310,4,True,77290,.\\Users\\acct.clerk01\\Documents\\Invoices,inv-1438.pdf,.pdf,184233,False,False,False,2026-03-11 15:02:44.9021133,2026-03-11 15:02:44.9021133,2026-03-11 15:02:45.0019224,2026-03-11 15:02:44.9021133,2026-03-11 15:02:45.0019224,2026-03-11 15:02:44.9021133,2026-03-13 14:20:31.7710422,2026-03-11 15:02:44.9021133\n'
    '88398,2,False,88396,.\\Users\\acct.clerk01\\AppData\\Local\\Microsoft\\Windows\\INetCache\\Content.Outlook\\K3Q9ZP1L,Invoice_0313.zip,.zip,210944,False,False,False,2026-03-13 16:41:52.6612004,2026-03-13 16:41:52.6612004,2026-03-13 16:41:52.7120391,2026-03-13 16:41:52.6612004,2026-03-13 16:48:03.1177420,2026-03-13 16:41:52.6612004,2026-03-13 16:41:55.0210933,2026-03-13 16:41:52.6612004\n'
    '61022,11,True,61000,.\\Users\\acct.clerk01\\Desktop,notes.txt,.txt,3120,False,False,False,2026-02-20 14:10:05.1203311,2026-02-20 14:10:05.1203311,2026-03-12 19:44:18.6630295,2026-02-20 14:10:05.1203311,2026-03-12 19:44:18.6630295,2026-02-20 14:10:05.1203311,2026-03-12 19:44:18.6630295,2026-02-20 14:10:05.1203311\n'
))
RESERVA['case-lab-p4/memory/windows.cmdline.txt'] = ''.join((
    '# SYNTHETIC TRAINING DATA (case LAB-P4). Resembles Volatility 3 windows.cmdline output for\n'
    '# fictional host WS-FIN-07. The encoded command decodes to a harmless placeholder string.\n'
    'PID\tProcess\tArgs\n'
    '4120\texplorer.exe\tC:\\Windows\\Explorer.EXE\n'
    '6632\tOUTLOOK.EXE\t"C:\\Program Files\\Microsoft Office\\root\\Office16\\OUTLOOK.EXE"\n'
    '7316\tpowershell.exe\tpowershell.exe -NoProfile -WindowStyle Hidden -EncodedCommand VwByAGkAdABlAC0ATwB1AHQAcAB1AHQAIAAnAEwAQQBCAC0AUAA0ACAAcwB5AG4AdABoAGUAdABpAGMAIABwAGwAYQBjAGUAaABvAGwAZABlAHIALAAgAG4AbwAgAHAAYQB5AGwAbwBhAGQAJwA=\n'
    '7488\tsynchelper.exe\t"C:\\Users\\acct.clerk01\\AppData\\Roaming\\SyncHelper\\synchelper.exe" --svc\n'
    '7704\trundll32.exe\tC:\\Windows\\system32\\rundll32.exe\n'
    '5124\tsvchost.exe\tC:\\Users\\Public\\svchost.exe\n'
    '6208\tmsedge.exe\t"C:\\Program Files (x86)\\Microsoft\\Edge\\Application\\msedge.exe"\n'
))
RESERVA['case-lab-p4/memory/windows.malfind.txt'] = ''.join((
    '# SYNTHETIC TRAINING DATA (case LAB-P4). Resembles Volatility 3 windows.malfind output for\n'
    '# fictional host WS-FIN-07. Hexdump bytes are illustrative only; nothing executable is here.\n'
    'PID\tProcess\tStart VPN\tEnd VPN\tTag\tProtection\tCommitCharge\tPrivateMemory\tFile output\tNotes\tHexdump\tDisasm\n'
    '\n'
    '7704\trundll32.exe\t0x1c0000\t0x1e0fff\tVadS\tPAGE_EXECUTE_READWRITE\t33\t1\tDisabled\tMZ header\n'
    '4d 5a 90 00 03 00 00 00 04 00 00 00 ff ff 00 00\tMZ..............\n'
    'b8 00 00 00 00 00 00 00 40 00 00 00 00 00 00 00\t........@.......\n'
    '00 00 00 00 00 00 00 00 00 00 00 00 00 00 00 00\t................\n'
    '00 00 00 00 00 00 00 00 00 00 00 00 f8 00 00 00\t................\n'
    '\n'
    '7316\tpowershell.exe\t0x7ff4a2c30000\t0x7ff4a2c3ffff\tVadS\tPAGE_EXECUTE_READWRITE\t2\t1\tDisabled\tN/A\n'
    '00 00 00 00 00 00 00 00 49 ba 60 1c 8d a2 f4 7f\t........I.`.....\n'
    '00 00 41 ff e3 cc cc cc cc cc cc cc cc cc cc cc\t..A.............\n'
    '49 ba 10 2d 8d a2 f4 7f 00 00 41 ff e3 cc cc cc\tI..-......A.....\n'
    'cc cc cc cc cc cc cc cc cc cc cc cc cc cc cc cc\t................\n'
    '\n'
    '6208\tmsedge.exe\t0x2b2c0040000\t0x2b2c007ffff\tVadS\tPAGE_EXECUTE_READWRITE\t64\t1\tDisabled\tN/A\n'
    '00 00 00 00 00 00 00 00 00 00 00 00 00 00 00 00\t................\n'
    '00 00 00 00 00 00 00 00 00 00 00 00 00 00 00 00\t................\n'
))
RESERVA['case-lab-p4/memory/windows.netscan.txt'] = ''.join((
    '# SYNTHETIC TRAINING DATA (case LAB-P4). Hand-written to resemble Volatility 3 windows.netscan\n'
    '# output for fictional host WS-FIN-07 (10.10.40.57). Public IPs are RFC 5737 documentation\n'
    '# addresses. Tab-separated.\n'
    'Offset\tProto\tLocalAddr\tLocalPort\tForeignAddr\tForeignPort\tState\tPID\tOwner\tCreated\n'
    '0xa3066fb3a010\tTCPv4\t0.0.0.0\t135\t0.0.0.0\t0\tLISTENING\t852\tsvchost.exe\t2026-03-13 12:58:04.000000 UTC\n'
    '0xa3066fb3a9e0\tTCPv4\t0.0.0.0\t445\t0.0.0.0\t0\tLISTENING\t4\tSystem\t2026-03-13 12:58:06.000000 UTC\n'
    '0xa3066fc10c20\tTCPv4\t0.0.0.0\t49664\t0.0.0.0\t0\tLISTENING\t672\tlsass.exe\t2026-03-13 12:58:04.000000 UTC\n'
    '0xa30671c90a60\tTCPv4\t10.10.40.57\t50344\t192.0.2.30\t443\tESTABLISHED\t6632\tOUTLOOK.EXE\t2026-03-14 04:58:12.000000 UTC\n'
    '0xa30671d99b20\tTCPv4\t10.10.40.57\t50402\t192.0.2.31\t443\tESTABLISHED\t6208\tmsedge.exe\t2026-03-14 05:02:40.000000 UTC\n'
    '0xa30672e9d8a0\tTCPv4\t10.10.40.57\t50811\t198.51.100.23\t443\tESTABLISHED\t7488\tsynchelper.exe\t2026-03-14 05:07:31.000000 UTC\n'
    '0xa30672e9f010\tTCPv4\t10.10.40.57\t50799\t198.51.100.23\t443\tCLOSED\t7488\tsynchelper.exe\t2026-03-14 05:02:30.000000 UTC\n'
    '0xa30672f55d30\tTCPv4\t10.10.40.57\t50822\t203.0.113.80\t8443\tSYN_SENT\t5124\tsvchost.exe\t2026-03-14 05:09:58.000000 UTC\n'
    '0xa30672f60a10\tUDPv4\t10.10.40.57\t55122\t*\t0\t\t7488\tsynchelper.exe\t2026-03-14 05:07:30.000000 UTC\n'
    '0xa3066fd0e2a0\tUDPv4\t0.0.0.0\t5353\t*\t0\t\t1100\tsvchost.exe\t2026-03-13 12:58:09.000000 UTC\n'
))
RESERVA['case-lab-p4/memory/windows.pstree.txt'] = ''.join((
    '# SYNTHETIC TRAINING DATA (case LAB-P4). Hand-written to resemble Volatility 3 windows.pstree\n'
    '# text output for a fictional host WS-FIN-07. No real memory image exists behind it. Columns\n'
    '# follow Volatility 3 2.x layout; exact columns vary by version. Tab-separated.\n'
    'PID\tPPID\tImageFileName\tOffset(V)\tThreads\tHandles\tSessionId\tWow64\tCreateTime\tExitTime\tAudit\tCmd\tPath\n'
    '4\t0\tSystem\t0xa3066c682040\t152\t-\tN/A\tFalse\t2026-03-13 12:58:01.000000 UTC\tN/A\t-\t-\t-\n'
    '* 92\t4\tRegistry\t0xa3066c6f1080\t4\t-\tN/A\tFalse\t2026-03-13 12:57:58.000000 UTC\tN/A\tRegistry\t-\t-\n'
    '* 348\t4\tsmss.exe\t0xa3066f0b4040\t2\t-\tN/A\tFalse\t2026-03-13 12:58:01.000000 UTC\tN/A\t\\Device\\HarddiskVolume3\\Windows\\System32\\smss.exe\t\\SystemRoot\\System32\\smss.exe\t\\SystemRoot\\System32\\smss.exe\n'
    '456\t444\tcsrss.exe\t0xa3066f6a3140\t11\t-\t0\tFalse\t2026-03-13 12:58:03.000000 UTC\tN/A\t\\Device\\HarddiskVolume3\\Windows\\System32\\csrss.exe\t%SystemRoot%\\system32\\csrss.exe ObjectDirectory=\\Windows SharedSection=1024,20480,768 Windows=On SubSystemType=Windows ServerDll=basesrv,1 ServerDll=winsrv:UserServerDllInitialization,3 ServerDll=sxssrv,4 ProfileControl=Off MaxRequestThreads=16\tC:\\Windows\\system32\\csrss.exe\n'
    '532\t444\twininit.exe\t0xa3066f6d8080\t1\t-\t0\tFalse\t2026-03-13 12:58:03.000000 UTC\tN/A\t\\Device\\HarddiskVolume3\\Windows\\System32\\wininit.exe\twininit.exe\tC:\\Windows\\system32\\wininit.exe\n'
    '* 644\t532\tservices.exe\t0xa3066f73a0c0\t6\t-\t0\tFalse\t2026-03-13 12:58:03.000000 UTC\tN/A\t\\Device\\HarddiskVolume3\\Windows\\System32\\services.exe\tC:\\Windows\\system32\\services.exe\tC:\\Windows\\system32\\services.exe\n'
    '** 800\t644\tsvchost.exe\t0xa3066f7d1080\t21\t-\t0\tFalse\t2026-03-13 12:58:04.000000 UTC\tN/A\t\\Device\\HarddiskVolume3\\Windows\\System32\\svchost.exe\tC:\\Windows\\system32\\svchost.exe -k DcomLaunch -p\tC:\\Windows\\system32\\svchost.exe\n'
    '** 852\t644\tsvchost.exe\t0xa3066f7f3080\t12\t-\t0\tFalse\t2026-03-13 12:58:04.000000 UTC\tN/A\t\\Device\\HarddiskVolume3\\Windows\\System32\\svchost.exe\tC:\\Windows\\system32\\svchost.exe -k RPCSS -p\tC:\\Windows\\system32\\svchost.exe\n'
    '** 1100\t644\tsvchost.exe\t0xa3066fa1e080\t38\t-\t0\tFalse\t2026-03-13 12:58:05.000000 UTC\tN/A\t\\Device\\HarddiskVolume3\\Windows\\System32\\svchost.exe\tC:\\Windows\\system32\\svchost.exe -k netsvcs -p\tC:\\Windows\\system32\\svchost.exe\n'
    '* 672\t532\tlsass.exe\t0xa3066f73e080\t9\t-\t0\tFalse\t2026-03-13 12:58:03.000000 UTC\tN/A\t\\Device\\HarddiskVolume3\\Windows\\System32\\lsass.exe\tC:\\Windows\\system32\\lsass.exe\tC:\\Windows\\system32\\lsass.exe\n'
    '540\t524\tcsrss.exe\t0xa3066f6c9140\t13\t-\t1\tFalse\t2026-03-13 12:58:03.000000 UTC\tN/A\t\\Device\\HarddiskVolume3\\Windows\\System32\\csrss.exe\t%SystemRoot%\\system32\\csrss.exe ObjectDirectory=\\Windows SharedSection=1024,20480,768 Windows=On SubSystemType=Windows ServerDll=basesrv,1 ServerDll=winsrv:UserServerDllInitialization,3 ServerDll=sxssrv,4 ProfileControl=Off MaxRequestThreads=16\tC:\\Windows\\system32\\csrss.exe\n'
    '548\t524\twinlogon.exe\t0xa3066f6d4080\t5\t-\t1\tFalse\t2026-03-13 12:58:03.000000 UTC\tN/A\t\\Device\\HarddiskVolume3\\Windows\\System32\\winlogon.exe\twinlogon.exe\tC:\\Windows\\system32\\winlogon.exe\n'
    '4120\t4088\texplorer.exe\t0xa30670a8c080\t74\t-\t1\tFalse\t2026-03-13 13:04:12.000000 UTC\tN/A\t\\Device\\HarddiskVolume3\\Windows\\explorer.exe\tC:\\Windows\\Explorer.EXE\tC:\\Windows\\Explorer.EXE\n'
    '* 6632\t4120\tOUTLOOK.EXE\t0xa30671c2a080\t48\t-\t1\tFalse\t2026-03-13 13:06:40.000000 UTC\tN/A\t\\Device\\HarddiskVolume3\\Program Files\\Microsoft Office\\root\\Office16\\OUTLOOK.EXE\t"C:\\Program Files\\Microsoft Office\\root\\Office16\\OUTLOOK.EXE" \tC:\\Program Files\\Microsoft Office\\root\\Office16\\OUTLOOK.EXE\n'
    '** 7316\t6632\tpowershell.exe\t0xa30672d10080\t14\t-\t1\tFalse\t2026-03-13 16:42:18.000000 UTC\tN/A\t\\Device\\HarddiskVolume3\\Windows\\System32\\WindowsPowerShell\\v1.0\\powershell.exe\tpowershell.exe -NoProfile -WindowStyle Hidden -EncodedCommand VwByAGkAdABlAC0ATwB1AHQAcAB1AHQAIAAnAEwAQQBCAC0AUAA0ACAAcwB5AG4AdABoAGUAdABpAGMAIABwAGwAYQBjAGUAaABvAGwAZABlAHIALAAgAG4AbwAgAHAAYQB5AGwAbwBhAGQAJwA=\tC:\\Windows\\System32\\WindowsPowerShell\\v1.0\\powershell.exe\n'
    '*** 7488\t7316\tsynchelper.exe\t0xa30672e44080\t9\t-\t1\tFalse\t2026-03-13 16:42:31.000000 UTC\tN/A\t\\Device\\HarddiskVolume3\\Users\\acct.clerk01\\AppData\\Roaming\\SyncHelper\\synchelper.exe\t"C:\\Users\\acct.clerk01\\AppData\\Roaming\\SyncHelper\\synchelper.exe" --svc\tC:\\Users\\acct.clerk01\\AppData\\Roaming\\SyncHelper\\synchelper.exe\n'
    '**** 7704\t7488\trundll32.exe\t0xa30672f1a080\t5\t-\t1\tFalse\t2026-03-13 16:43:02.000000 UTC\tN/A\t\\Device\\HarddiskVolume3\\Windows\\System32\\rundll32.exe\tC:\\Windows\\system32\\rundll32.exe\tC:\\Windows\\system32\\rundll32.exe\n'
    '* 5124\t4120\tsvchost.exe\t0xa30671f03080\t3\t-\t1\tFalse\t2026-03-13 16:44:10.000000 UTC\tN/A\t\\Device\\HarddiskVolume3\\Users\\Public\\svchost.exe\tC:\\Users\\Public\\svchost.exe\tC:\\Users\\Public\\svchost.exe\n'
    '* 6208\t4120\tmsedge.exe\t0xa30671d7b080\t41\t-\t1\tFalse\t2026-03-13 13:10:55.000000 UTC\tN/A\t\\Device\\HarddiskVolume3\\Program Files (x86)\\Microsoft\\Edge\\Application\\msedge.exe\t"C:\\Program Files (x86)\\Microsoft\\Edge\\Application\\msedge.exe"\tC:\\Program Files (x86)\\Microsoft\\Edge\\Application\\msedge.exe\n'
))

EVID_2_3 = {}           # o que você vai analisar de fato (nome curto -> texto)
ORIGEM = {}              # de onde veio cada arquivo: "rede" ou "reserva"

for nome, reserva in RESERVA.items():
    texto = None
    if not FORCAR_OFFLINE:
        try:
            # timeout curto: se a rede não responder em 8 s, seguimos com o plano B
            with urllib.request.urlopen(URL_BASE + nome, timeout=8) as resp:
                texto = resp.read().decode("utf-8")
            ORIGEM[nome] = "rede"
        except Exception as erro:            # sem internet, proxy, 404... qualquer falha
            print(f"⚠️  {nome.split('/')[-1]}: rede indisponível ({type(erro).__name__}), usando a cópia embutida")
    if texto is None:
        texto = reserva
        ORIGEM[nome] = "reserva"
    EVID_2_3[nome.split("/")[-1]] = texto   # guardamos pelo nome curto
    h = hashlib.sha256(texto.encode("utf-8")).hexdigest()
    # Se o upstream mudou depois que este notebook foi gerado, os hashes divergem: avisamos.
    igual = "= cópia embutida" if texto == reserva else "≠ cópia embutida (o upstream mudou!)"
    print(f"{ORIGEM[nome]:8} {h[:16]}…  {nome.split('/')[-1]}  ({igual})")


In [ ]:
GRUPOS = {
    "EVID-002": ["ws-fin-07-security.csv", "ws-fin-07-mft-excerpt.csv"],
    "EVID-003": ["windows.cmdline.txt", "windows.malfind.txt", "windows.netscan.txt", "windows.pstree.txt"],
}
for item, nomes in GRUPOS.items():
    pasta = BASE / "evidence" / item
    pasta.mkdir(exist_ok=True)
    for n in nomes:
        destino = pasta / n
        if destino.exists():
            destino.chmod(stat.S_IRUSR | stat.S_IWUSR)
        with open(destino, "w", encoding="utf-8", newline="") as f:
            f.write(EVID_2_3[n])
    so_leitura(pasta)
    print(f"--- {item}")
    gerar_manifesto(pasta, BASE / "notes" / f"{item}.receipt.sha256")
    registrar_acao(f"received {item}")


## 7. ➕ O EVID-SA-001 da Serra Azul

Mesma disciplina, outro caso. O pen drive tem três arquivos: proxy, firewall e webmail. Estes dados são deste repositório (sintéticos), então a célula tenta o GitHub do **investigador-cyber-90-dias** e cai para a cópia embutida.


In [ ]:
# ------------------------------------------------------------
# RECEBER O EVID-SA-001 (PEN DRIVE DA SERRA AZUL)
# Plano A: baixar do repositório original (GitHub raw), com urllib.
# Plano B: usar a cópia embutida logo abaixo (o arquivo inteiro,
#          copiado do upstream/), para o notebook rodar sem internet.
# ------------------------------------------------------------
import urllib.request      # baixa arquivos da web, sem instalar nada
import hashlib             # SHA-256: a impressão digital dos bytes

FORCAR_OFFLINE = False     # True = nem tenta a rede (útil em sala sem Wi-Fi)
URL_BASE = 'https://raw.githubusercontent.com/Ridd1kulusC0d3r/investigador-cyber-90-dias/main/caso-continuo/dados/'

RESERVA = {}             # caminho do arquivo -> conteúdo embutido (plano B)
RESERVA['dia-49/EVID-SA-001/proxy-sa.log'] = ''.join((
    'Oct  5 11:13:25 proxy-sa squid[2211]: 10.20.40.57 cx.centro07 CONNECT portal-noticias.example:443 200 out=3547 in=65569 dst=192.0.2.25\n'
    'Oct  5 11:18:47 proxy-sa squid[2211]: 10.20.40.55 atd.centro02 CONNECT portal-noticias.example:443 200 out=3123 in=73679 dst=192.0.2.11\n'
    'Oct  5 11:19:38 proxy-sa squid[2211]: 10.20.40.57 cx.centro07 GET http://lnk.example/sa7 302 out=412 in=380 dst=198.51.100.201\n'
    'Oct  5 11:19:41 proxy-sa squid[2211]: 10.20.40.57 cx.centro07 CONNECT serraazul-pix.example:443 200 out=1840 in=48120 dst=203.0.113.77\n'
    'Oct  5 11:21:05 proxy-sa squid[2211]: 10.20.40.57 cx.centro07 CONNECT serraazul-pix.example:443 200 out=2311 in=5120 dst=203.0.113.77\n'
    'Oct  5 11:21:52 proxy-sa squid[2211]: 10.20.40.57 cx.centro07 CONNECT serraazul-pix.example:443 200 out=2290 in=4980 dst=203.0.113.77\n'
    'Oct  5 11:23:12 proxy-sa squid[2211]: 10.20.40.57 cx.centro07 CONNECT serraazul-pix.example:443 200 out=1622 in=412880 dst=203.0.113.77\n'
    'Oct  5 11:25:02 proxy-sa squid[2211]: 10.20.40.57 cx.centro07 CONNECT atualiza-modulo.example:443 200 out=1198 in=796 dst=198.51.100.140\n'
    'Oct  5 11:26:02 proxy-sa squid[2211]: 10.20.40.55 atd.centro02 CONNECT busca.example:443 200 out=2799 in=72449 dst=192.0.2.27\n'
    'Oct  5 11:29:54 proxy-sa squid[2211]: 10.20.40.57 cx.centro07 CONNECT atualiza-modulo.example:443 200 out=1227 in=630 dst=198.51.100.140\n'
    'Oct  5 11:34:49 proxy-sa squid[2211]: 10.20.40.51 cx.centro05 CONNECT portal-noticias.example:443 200 out=3094 in=55049 dst=192.0.2.35\n'
    'Oct  5 11:34:56 proxy-sa squid[2211]: 10.20.40.57 cx.centro07 CONNECT atualiza-modulo.example:443 200 out=1543 in=741 dst=198.51.100.140\n'
    'Oct  5 11:39:41 proxy-sa squid[2211]: 10.20.40.53 cx.centro06 CONNECT intranet-cdn.example:443 200 out=2473 in=4780 dst=192.0.2.16\n'
    'Oct  5 11:39:47 proxy-sa squid[2211]: 10.20.40.57 cx.centro07 CONNECT atualiza-modulo.example:443 200 out=1279 in=771 dst=198.51.100.140\n'
    'Oct  5 11:43:55 proxy-sa squid[2211]: 10.20.40.51 cx.centro05 CONNECT mapas.example:443 200 out=2981 in=8490 dst=192.0.2.16\n'
    'Oct  5 11:44:51 proxy-sa squid[2211]: 10.20.40.57 cx.centro07 CONNECT atualiza-modulo.example:443 200 out=1303 in=722 dst=198.51.100.140\n'
    'Oct  5 11:48:08 proxy-sa squid[2211]: 10.20.40.53 cx.centro06 CONNECT mapas.example:443 200 out=1085 in=61379 dst=192.0.2.23\n'
    'Oct  5 11:49:50 proxy-sa squid[2211]: 10.20.40.57 cx.centro07 CONNECT atualiza-modulo.example:443 200 out=1525 in=806 dst=198.51.100.140\n'
    'Oct  5 11:54:59 proxy-sa squid[2211]: 10.20.40.57 cx.centro07 CONNECT atualiza-modulo.example:443 200 out=1597 in=721 dst=198.51.100.140\n'
    'Oct  5 11:57:59 proxy-sa squid[2211]: 10.20.40.52 ger.centro01 CONNECT busca.example:443 200 out=2448 in=28809 dst=192.0.2.12\n'
    'Oct  5 11:59:52 proxy-sa squid[2211]: 10.20.40.57 cx.centro07 CONNECT atualiza-modulo.example:443 200 out=1477 in=826 dst=198.51.100.140\n'
    'Oct  5 12:04:45 proxy-sa squid[2211]: 10.20.40.57 cx.centro07 CONNECT atualiza-modulo.example:443 200 out=1323 in=899 dst=198.51.100.140\n'
    'Oct  5 12:07:10 proxy-sa squid[2211]: 10.20.40.57 cx.centro07 CONNECT atualiza-modulo.example:443 200 out=3145728 in=1022 dst=198.51.100.140\n'
    'Oct  5 12:09:42 proxy-sa squid[2211]: 10.20.40.57 cx.centro07 CONNECT atualiza-modulo.example:443 200 out=1106 in=629 dst=198.51.100.140\n'
    'Oct  5 12:14:35 proxy-sa squid[2211]: 10.20.40.52 ger.centro01 CONNECT busca.example:443 200 out=1480 in=5140 dst=192.0.2.22\n'
    'Oct  5 12:14:49 proxy-sa squid[2211]: 10.20.40.57 cx.centro07 CONNECT atualiza-modulo.example:443 200 out=1277 in=700 dst=198.51.100.140\n'
    'Oct  5 12:17:49 proxy-sa squid[2211]: 10.20.40.52 ger.centro01 CONNECT intranet-cdn.example:443 200 out=2959 in=37419 dst=192.0.2.16\n'
    'Oct  5 12:19:58 proxy-sa squid[2211]: 10.20.40.57 cx.centro07 CONNECT atualiza-modulo.example:443 200 out=1350 in=709 dst=198.51.100.140\n'
    'Oct  5 12:24:59 proxy-sa squid[2211]: 10.20.40.57 cx.centro07 CONNECT atualiza-modulo.example:443 200 out=1585 in=627 dst=198.51.100.140\n'
    'Oct  5 12:26:36 proxy-sa squid[2211]: 10.20.40.52 ger.centro01 CONNECT fornecedor-cartoes.example:443 200 out=1070 in=32479 dst=192.0.2.24\n'
    'Oct  5 12:30:03 proxy-sa squid[2211]: 10.20.40.57 cx.centro07 CONNECT atualiza-modulo.example:443 200 out=1492 in=730 dst=198.51.100.140\n'
    'Oct  5 12:30:32 proxy-sa squid[2211]: 10.20.40.51 cx.centro05 CONNECT portal-noticias.example:443 200 out=2525 in=5542 dst=192.0.2.30\n'
    'Oct  5 12:35:05 proxy-sa squid[2211]: 10.20.40.57 cx.centro07 CONNECT atualiza-modulo.example:443 200 out=1101 in=854 dst=198.51.100.140\n'
    'Oct  5 12:36:01 proxy-sa squid[2211]: 10.20.40.53 cx.centro06 CONNECT mapas.example:443 200 out=2206 in=34292 dst=192.0.2.18\n'
))
RESERVA['dia-49/EVID-SA-001/fw-serraazul.log'] = ''.join((
    '2026-10-05 08:13:25 fw-sa ACCEPT TCP 10.20.40.57:57000 -> 192.0.2.25:443 rule=allow-https-out bytes=69116\n'
    '2026-10-05 08:18:47 fw-sa ACCEPT TCP 10.20.40.55:53716 -> 192.0.2.11:443 rule=allow-https-out bytes=76802\n'
    '2026-10-05 08:19:38 fw-sa ACCEPT TCP 10.20.40.57:51018 -> 198.51.100.201:80 rule=allow-http-out bytes=792\n'
    '2026-10-05 08:19:41 fw-sa ACCEPT TCP 10.20.40.57:51022 -> 203.0.113.77:443 rule=allow-https-out bytes=49960\n'
    '2026-10-05 08:21:05 fw-sa ACCEPT TCP 10.20.40.57:51031 -> 203.0.113.77:443 rule=allow-https-out bytes=7431\n'
    '2026-10-05 08:21:52 fw-sa ACCEPT TCP 10.20.40.57:51036 -> 203.0.113.77:443 rule=allow-https-out bytes=7270\n'
    '2026-10-05 08:23:12 fw-sa ACCEPT TCP 10.20.40.57:51040 -> 203.0.113.77:443 rule=allow-https-out bytes=414502\n'
    '2026-10-05 08:25:02 fw-sa ACCEPT TCP 10.20.40.57:51100 -> 198.51.100.140:443 rule=allow-https-out bytes=1994\n'
    '2026-10-05 08:26:02 fw-sa ACCEPT TCP 10.20.40.55:52944 -> 192.0.2.27:443 rule=allow-https-out bytes=75248\n'
    '2026-10-05 08:29:54 fw-sa ACCEPT TCP 10.20.40.57:51102 -> 198.51.100.140:443 rule=allow-https-out bytes=1857\n'
    '2026-10-05 08:34:49 fw-sa ACCEPT TCP 10.20.40.51:56158 -> 192.0.2.35:443 rule=allow-https-out bytes=58143\n'
    '2026-10-05 08:34:56 fw-sa ACCEPT TCP 10.20.40.57:51111 -> 198.51.100.140:443 rule=allow-https-out bytes=2284\n'
    '2026-10-05 08:39:41 fw-sa ACCEPT TCP 10.20.40.53:59046 -> 192.0.2.16:443 rule=allow-https-out bytes=7253\n'
    '2026-10-05 08:39:47 fw-sa ACCEPT TCP 10.20.40.57:51120 -> 198.51.100.140:443 rule=allow-https-out bytes=2050\n'
    '2026-10-05 08:43:55 fw-sa ACCEPT TCP 10.20.40.51:49596 -> 192.0.2.16:443 rule=allow-https-out bytes=11471\n'
    '2026-10-05 08:44:51 fw-sa ACCEPT TCP 10.20.40.57:51125 -> 198.51.100.140:443 rule=allow-https-out bytes=2025\n'
    '2026-10-05 08:48:08 fw-sa ACCEPT TCP 10.20.40.53:59292 -> 192.0.2.23:443 rule=allow-https-out bytes=62464\n'
    '2026-10-05 08:49:50 fw-sa ACCEPT TCP 10.20.40.57:51129 -> 198.51.100.140:443 rule=allow-https-out bytes=2331\n'
    '2026-10-05 08:54:59 fw-sa ACCEPT TCP 10.20.40.57:51132 -> 198.51.100.140:443 rule=allow-https-out bytes=2318\n'
    '2026-10-05 08:57:59 fw-sa ACCEPT TCP 10.20.40.52:56972 -> 192.0.2.12:443 rule=allow-https-out bytes=31257\n'
    '2026-10-05 08:59:52 fw-sa ACCEPT TCP 10.20.40.57:51139 -> 198.51.100.140:443 rule=allow-https-out bytes=2303\n'
    '2026-10-05 09:04:45 fw-sa ACCEPT TCP 10.20.40.57:51147 -> 198.51.100.140:443 rule=allow-https-out bytes=2222\n'
    '2026-10-05 09:07:10 fw-sa ACCEPT TCP 10.20.40.57:51377 -> 198.51.100.140:443 rule=allow-https-out bytes=3150210\n'
    '2026-10-05 09:09:42 fw-sa ACCEPT TCP 10.20.40.57:51154 -> 198.51.100.140:443 rule=allow-https-out bytes=1735\n'
    '2026-10-05 09:14:35 fw-sa ACCEPT TCP 10.20.40.52:49642 -> 192.0.2.22:443 rule=allow-https-out bytes=6620\n'
    '2026-10-05 09:14:49 fw-sa ACCEPT TCP 10.20.40.57:51157 -> 198.51.100.140:443 rule=allow-https-out bytes=1977\n'
    '2026-10-05 09:17:49 fw-sa ACCEPT TCP 10.20.40.52:49705 -> 192.0.2.16:443 rule=allow-https-out bytes=40378\n'
    '2026-10-05 09:19:58 fw-sa ACCEPT TCP 10.20.40.57:51161 -> 198.51.100.140:443 rule=allow-https-out bytes=2059\n'
    '2026-10-05 09:24:59 fw-sa ACCEPT TCP 10.20.40.57:51166 -> 198.51.100.140:443 rule=allow-https-out bytes=2212\n'
    '2026-10-05 09:26:36 fw-sa ACCEPT TCP 10.20.40.52:52879 -> 192.0.2.24:443 rule=allow-https-out bytes=33549\n'
    '2026-10-05 09:30:03 fw-sa ACCEPT TCP 10.20.40.57:51169 -> 198.51.100.140:443 rule=allow-https-out bytes=2222\n'
    '2026-10-05 09:30:32 fw-sa ACCEPT TCP 10.20.40.51:53871 -> 192.0.2.30:443 rule=allow-https-out bytes=8067\n'
    '2026-10-05 09:31:02 fw-sa ACCEPT TCP 203.0.113.91:52764 -> 10.20.5.20:443 rule=allow-webmail-in bytes=6120\n'
    '2026-10-05 09:31:15 fw-sa ACCEPT TCP 203.0.113.91:56244 -> 10.20.5.20:443 rule=allow-webmail-in bytes=6120\n'
    '2026-10-05 09:31:40 fw-sa ACCEPT TCP 203.0.113.91:47544 -> 10.20.5.20:443 rule=allow-webmail-in bytes=6120\n'
    '2026-10-05 09:32:05 fw-sa ACCEPT TCP 203.0.113.91:52211 -> 10.20.5.20:443 rule=allow-webmail-in bytes=88412\n'
    '2026-10-05 09:35:05 fw-sa ACCEPT TCP 10.20.40.57:51177 -> 198.51.100.140:443 rule=allow-https-out bytes=1955\n'
    '2026-10-05 09:36:01 fw-sa ACCEPT TCP 10.20.40.53:58780 -> 192.0.2.18:443 rule=allow-https-out bytes=36498\n'
))
RESERVA['dia-49/EVID-SA-001/webmail-sa.jsonl'] = ''.join((
    '{"ts": "2026-10-05T10:53:02Z", "event": "auth.login", "user": "cx.centro07", "src_ip": "10.20.40.57", "result": "success"}\n'
    '{"ts": "2026-10-05T10:54:41Z", "event": "auth.login", "user": "ger.centro01", "src_ip": "10.20.40.52", "result": "success"}\n'
    '{"ts": "2026-10-05T12:31:02Z", "event": "auth.login", "user": "cx.centro07", "src_ip": "203.0.113.91", "result": "failure"}\n'
    '{"ts": "2026-10-05T12:31:15Z", "event": "auth.login", "user": "cx.centro07", "src_ip": "203.0.113.91", "result": "failure"}\n'
    '{"ts": "2026-10-05T12:31:40Z", "event": "auth.login", "user": "cx.centro07", "src_ip": "203.0.113.91", "result": "failure"}\n'
    '{"ts": "2026-10-05T12:32:05Z", "event": "auth.login", "user": "cx.centro07", "src_ip": "203.0.113.91", "result": "success"}\n'
    '{"ts": "2026-10-05T12:33:50Z", "event": "mailbox.rule_create", "user": "cx.centro07", "src_ip": "203.0.113.91", "result": "success"}\n'
    '{"ts": "2026-10-05T13:05:01Z", "event": "auth.login", "user": "ger.centro01", "src_ip": "10.20.40.52", "result": "failure"}\n'
    '{"ts": "2026-10-05T13:05:19Z", "event": "auth.login", "user": "ger.centro01", "src_ip": "10.20.40.52", "result": "success"}\n'
))

EVID_SA = {}           # o que você vai analisar de fato (nome curto -> texto)
ORIGEM = {}              # de onde veio cada arquivo: "rede" ou "reserva"

for nome, reserva in RESERVA.items():
    texto = None
    if not FORCAR_OFFLINE:
        try:
            # timeout curto: se a rede não responder em 8 s, seguimos com o plano B
            with urllib.request.urlopen(URL_BASE + nome, timeout=8) as resp:
                texto = resp.read().decode("utf-8")
            ORIGEM[nome] = "rede"
        except Exception as erro:            # sem internet, proxy, 404... qualquer falha
            print(f"⚠️  {nome.split('/')[-1]}: rede indisponível ({type(erro).__name__}), usando a cópia embutida")
    if texto is None:
        texto = reserva
        ORIGEM[nome] = "reserva"
    EVID_SA[nome.split("/")[-1]] = texto   # guardamos pelo nome curto
    h = hashlib.sha256(texto.encode("utf-8")).hexdigest()
    # Se o upstream mudou depois que este notebook foi gerado, os hashes divergem: avisamos.
    igual = "= cópia embutida" if texto == reserva else "≠ cópia embutida (o upstream mudou!)"
    print(f"{ORIGEM[nome]:8} {h[:16]}…  {nome.split('/')[-1]}  ({igual})")


In [ ]:
ISCA = Path("caso-isca")
for sub in ("evidence", "work", "notes"):
    (ISCA / sub).mkdir(parents=True, exist_ok=True)
ORIG_SA = ISCA / "evidence" / "EVID-SA-001"
ORIG_SA.mkdir(exist_ok=True)
for nome, texto in EVID_SA.items():
    destino = ORIG_SA / nome
    if destino.exists():
        destino.chmod(stat.S_IRUSR | stat.S_IWUSR)
    with open(destino, "w", encoding="utf-8", newline="") as f:
        f.write(texto)
so_leitura(ORIG_SA)

LOG_SA = ISCA / "notes" / "actions.log"
registrar_acao("recebido EVID-SA-001, lacre SA-0001 intacto, entregue pela TI da Serra Azul", LOG_SA)
registrar_acao("lacre SA-0001 rompido por: analista Vigia (você), finalidade: cópia; novo lacre VIGIA-0107", LOG_SA)
MAN_SA = gerar_manifesto(ORIG_SA, ISCA / "notes" / "EVID-SA-001.receipt.sha256")
COPIA_SA = ISCA / "work" / "EVID-SA-001"
if COPIA_SA.exists():
    shutil.rmtree(COPIA_SA)
shutil.copytree(ORIG_SA, COPIA_SA)
print()
assert conferir(COPIA_SA, MAN_SA) == 0
registrar_acao("cópia de trabalho work/EVID-SA-001 conferida (3 OK)", LOG_SA)


> 🧠 **Camada 8.** Repare que o log de ações registra o rompimento do lacre *no momento*. Daqui a três semanas, a sua memória de "rompi o lacre depois de calcular o hash" vai ser uma reconstrução (Loftus). O log, não.

## 8. 🎮 Quiz: as 10 etapas do art. 158-B e a ordem de volatilidade

Duas rodadas. Na primeira, você diz **qual etapa vem a seguir**. Na segunda, entre dois tipos de evidência, **qual você coleta primeiro** num sistema ligado (RFC 3227).

Quer só ler o gabarito? `MODO_JOGO = False`.


In [ ]:
MODO_JOGO = True   # False = mostra só o gabarito

# As 10 etapas do CPP, art. 158-B, na ordem da lei.
ETAPAS = ["reconhecimento", "isolamento", "fixação", "coleta", "acondicionamento",
          "transporte", "recebimento", "processamento", "armazenamento", "descarte"]

# O que cada etapa significa NO CASO SERRA AZUL (exemplo, não texto da lei).
NO_CASO = {
    "reconhecimento": "a TI percebe que os logs da manhã de 5/out interessam ao caso",
    "isolamento": "ninguém mais mexe no notebook SA-CX-07 nem nos logs originais",
    "fixação": "descrição e prints de como os logs e o notebook foram encontrados",
    "coleta": "exportação dos logs para o pen drive",
    "acondicionamento": "pen drive em envelope com lacre numerado SA-0001",
    "transporte": "a TI leva o envelope até a Vigia",
    "recebimento": "você registra quem entregou, quando (UTC) e o lacre",
    "processamento": "cópia de trabalho conferida e análise",
    "armazenamento": "original em local controlado, com quem tem acesso registrado",
    "descarte": "devolução, entrega à autoridade ou destruição autorizada",
}

acertos = total = 0
print("RODADA 1 — qual é a próxima etapa? (digite o número)")
for i in range(len(ETAPAS) - 1):
    atual, certa = ETAPAS[i], ETAPAS[i + 1]
    # opções: a certa + duas outras, numa ordem fixa (alfabética) para não depender de sorte
    opcoes = sorted({certa, ETAPAS[(i + 3) % 10], ETAPAS[(i + 6) % 10]})
    print(f"\nDepois de '{atual}' vem:")
    for k, o in enumerate(opcoes, 1):
        print(f"  {k}. {o}")
    if MODO_JOGO:
        r = input("  Sua resposta: ").strip()
        ok = r.isdigit() and 1 <= int(r) <= len(opcoes) and opcoes[int(r) - 1] == certa
        acertos += ok
        total += 1
        print("  ✅" if ok else "  ❌", end=" ")
    print(f"Resposta: {certa} — no caso: {NO_CASO[certa]}")

VOLATIL = ["registradores e cache da CPU", "tabela de roteamento, ARP, processos e memória",
           "sistemas de arquivos temporários", "disco", "logs remotos e monitoramento",
           "configuração física e topologia", "mídia de arquivamento"]
PARES = [(3, 1), (0, 6), (4, 2), (1, 3), (5, 4)]   # (índice A, índice B) em VOLATIL
print("\nRODADA 2 — sistema ligado: o que você coleta PRIMEIRO? (a ou b)")
for a, b in PARES:
    print(f"\n  a) {VOLATIL[a]}\n  b) {VOLATIL[b]}")
    certa = "a" if a < b else "b"
    if MODO_JOGO:
        r = input("  Sua resposta: ").strip().lower()
        ok = r == certa
        acertos += ok
        total += 1
        print("  ✅" if ok else "  ❌", end=" ")
    print(f"Resposta: {certa} ({VOLATIL[min(a, b)]} some primeiro)")

if MODO_JOGO:
    print(f"\n🏁 {acertos}/{total}. Checkpoint BR: saber as 10 etapas na ordem sem consultar.")


## 9. 📝 O formulário de custódia (Partes A a E)

O formulário do original, traduzido, mais a ➕ Parte E com os campos que o CPP pede. Os valores padrão são do **EVID-SA-001**. A célula preenche as transferências 1 e 2 com os hashes que você acabou de calcular e grava `caso-isca/notes/custodia-EVID-SA-001.md`.


In [ ]:
#@title Formulário de cadeia de custódia { display-mode: "form" }
caso = "VIGIA-2026-001 / Operação Isca"  #@param {type:"string"}
item = "EVID-SA-001"  #@param {type:"string"}
descricao = "Pen drive 16 GB, preto, etiqueta 'TI-SA 03', com 3 arquivos de log"  #@param {type:"string"}
coletado_de = "Exportação dos servidores de proxy, firewall e webmail da Cooperativa Serra Azul"  #@param {type:"string"}
autoridade = "Contrato Vigia x Serra Azul (Marta Quintão, 05/10/2026 11h00 BRT) + política de uso de equipamentos da cooperativa"  #@param {type:"string"}
coletado_por = "Analista de TI da Cooperativa Serra Azul"  #@param {type:"string"}
coleta_utc = "2026-10-05T18:30:00Z (15h30 America/Sao_Paulo)"  #@param {type:"string"}
lacre = "SA-0001"  #@param {type:"string"}
recebido_utc = "2026-10-05T19:05:00Z (16h05 America/Sao_Paulo)"  #@param {type:"string"}
novo_lacre = "VIGIA-0107"  #@param {type:"string"}
examinador = "Você, analista de investigação júnior, Vigia"  #@param {type:"string"}

linhas_hash = Path(MAN_SA).read_text(encoding="utf-8").strip().splitlines()
hash_lista = "<br>".join(l.replace("  ", " ") for l in linhas_hash)

form = f"""# Formulário de cadeia de custódia — {item}

Caso fictício (Operação Isca). Nunca edite uma linha passada; corrija com uma linha nova.

## Parte A: identificação do item

| Campo | Registro |
|---|---|
| Número do caso | {caso} |
| Número do item de evidência | {item} |
| Descrição | {descricao} |
| Número de série | (anotar do dispositivo) |
| Coletado de | {coletado_de} |
| Autoridade legal para a coleta | {autoridade} |
| Coletado por | {coletado_por} |
| Data e hora da coleta (UTC e fuso local) | {coleta_utc} |
| Condição na coleta | Pen drive em envelope pardo lacrado, sem danos aparentes |
| Fotografias | (nomes dos arquivos e hashes) |
| Embalagem e número do lacre | Envelope pardo, lacre {lacre} |

## Parte B: registro de aquisição

| Campo | Registro |
|---|---|
| Bloqueador de escrita | Software: cópia para pasta somente leitura (chmod a-w) |
| Ferramenta e versão | Python hashlib (notebook do Dia 49) |
| Comando exato | gerar_manifesto(evidence/{item}) |
| Hash de origem, algoritmo | SHA-256, por arquivo: {hash_lista} |
| Arquivos e formato | 3 arquivos de texto (log, log, jsonl) |
| Hash da cópia, algoritmo | SHA-256, conferido contra o manifesto |
| Os hashes conferem? | sim (3 OK) |
| Erros de leitura | nenhum |
| Início e fim (UTC) | (preencha com o actions.log) |
| Assinatura do examinador | {examinador} |

## Parte C: registro de transferências

| # | Data e hora (UTC) | Entregue por | Recebido por | Finalidade | Local após | Lacre intacto? | Hash reconferido |
|---|---|---|---|---|---|---|---|
| 1 | {recebido_utc} | {coletado_por} | {examinador} | Análise do caso | Sala de evidências da Vigia | sim ({lacre}) | n/a, lacrado |
| 2 | (actions.log) | {examinador} | {examinador} | Cópia de trabalho | caso-isca/work/{item} | rompido; novo lacre {novo_lacre} | sim, 3 OK |

## Parte D: destinação final

| Campo | Registro |
|---|---|
| Destinação e autoridade | (a definir com Marta Quintão e Dr. Hélio Brandão) |
| Data e hora (UTC) | |
| Testemunha | |

## ➕ Parte E: campos que o CPP pede (arts. 158-A a 158-F)

| Campo | Registro |
|---|---|
| Etapa atual (art. 158-B) | recebimento → processamento |
| Quem reconheceu o vestígio | TI da Serra Azul, ao ligar o clique da caixa ao SMS do golpe |
| Fixação | (prints e descrição dos servidores de origem) |
| Recipiente e lacre numerado (art. 158-D) | envelope pardo, lacre {lacre} |
| Rompimento de lacre (art. 158-D, §4º) | {examinador}; {recebido_utc}; Vigia; finalidade: cópia de trabalho; novo lacre {novo_lacre} |
| Lacre rompido guardado no novo recipiente? (§5º) | sim |
| Recebimento (art. 158-B, VII) | {recebido_utc}, entregue por {coletado_por} |
| Armazenamento e acesso | cofre da Vigia; acesso: você e Rafa |
| Destino previsto (art. 158-B, X) | entrega à autoridade policial via cooperativa, se houver notícia-crime |
"""

destino = ISCA / "notes" / f"custodia-{item}.md"
destino.write_text(form, encoding="utf-8")
print(form)
print("💾", destino, "\n🔏 SHA-256 do formulário:", sha256_arquivo(destino))
registrar_acao(f"formulário de custódia de {item} gravado, sha256={sha256_arquivo(destino)[:16]}…", LOG_SA)

try:
    from google.colab import files       # no Colab, baixa o formulário e o manifesto
    files.download(str(destino))
    files.download(str(MAN_SA))
except ImportError:
    print("(Fora do Colab: os arquivos ficaram em caso-isca/notes/)")


## 10. O que ficou no `actions.log`


In [ ]:
for log in (BASE / "notes" / "actions.log", LOG_SA):
    print(f"--- {log}")
    print(log.read_text(encoding="utf-8"))


## ✅ Antes de fechar o Dia 49

- [ ] `EVID-001.receipt.sha256` com 4 linhas e a cópia conferida (4 OK)
- [ ] `actions.log` com o teste de adulteração **e** com a substituição da cópia
- [ ] manifestos do EVID-002 e do EVID-003
- [ ] `EVID-SA-001.receipt.sha256` (3 OK) e o formulário com a Parte E
- [ ] as 10 etapas do art. 158-B, de cor
- [ ] em duas frases: por que um hash só, depois da análise, vale menos que dois (antes e depois)?

**Amanhã (Dia 50):** imagem de disco bit a bit. Você vai copiar um "pen drive" setor por setor e provar que a imagem é igual à origem.

---
<sub>Baseado em cyber-investigator-skills por ThiefDroppers (MIT) · Investigador Cyber 90 Dias por Deivison Lourenço</sub>
